> **Research record.** Cleaned copy of `800-research-users-locations/3001-explore-third-places-opus.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Descriptive statistics of the detected third places and of their visitors' behaviour.
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

# 3001 — Detected third places: descriptive statistics and visitor behaviour

Explores the PU-model output `phd_models_results.google_3rd_places_detected_pu_20260601`
(places predicted to be third places) and tests, in the tradition of the classic third-place
literature (Oldenburg 1989), how **remote workers** use these places compared with **on-site workers**.

Structure:

1. **Place level** — what kinds of places were detected, where they sit in the urban fabric.
2. **All workers vs remote workers** — place-level visit metrics from `pois_visits_metrics_2022_2023`.
3. **Location context** — Tel Aviv-Yafo (`semel_yishuv` = 5000) versus the rest of Gush Dan, and same-category POI density within 1 km.
4. **Visit level** — hourly rhythm, distance to home, visit duration, arrivals from home
   (dwells joined to `user_profiles_monthly`).
5. **Figures A–F** and a summary of findings.

The PU table is *place level* (one row per POI, no visits), so everything by hour is built by
joining `phd_dwells.dwells_pois_home_work_v2` to the detected places and to monthly user profiles.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from geofunctions import utils

In [ ]:
PU_TABLE = "phd-habidatum.phd_models_results.google_3rd_places_detected_pu_20260601"
POI_UNIVERSE_TABLE = "phd-habidatum.phd_pois.google_3rdplaces_cities"
VISIT_METRICS_TABLE = "phd-habidatum.phd_datasets.pois_visits_metrics_2022_2023"
DWELLS_TABLE = "phd-habidatum.phd_dwells.dwells_pois_home_work_v2"
USER_PROFILES_TABLE = "phd-habidatum.phd_datasets.user_profiles_monthly"

CENSUS_TABLE = "phd-habidatum.phd_census.all_sa_main_features"
CENSUS_2022_TABLE = "phd-habidatum.phd_census.census_2022"

START_PERIOD = "2022-01-01"
END_PERIOD = "2023-12-01"

SHORT_VISIT_HOURS = 10 / 60  # short visit: under 10 minutes
LONG_VISIT_HOURS = 1         # long visit: more than one hour; medium is [10 min, 1 h]
FROM_PREV_MAX_GAP_H = 2      # max hours since the previous dwell to call it an arrival from home
DENSITY_RADIUS_M = 1000      # radius for the same-category POI count
# Hour windows match 005-third-places-users-signals / pois_visits_metrics time_period
# (SQL BETWEEN is inclusive: lo:00–hi:59).
MORNING_HOURS = (6, 9)                 # 06:00–09:59
EARLY_AFTERNOON_HOURS = (10, 13)       # 10:00–13:59
LATE_AFTERNOON_HOURS = (14, 18)        # 14:00–18:59
EVENING_HOURS = (19, 22)               # 19:00–22:59
DAYTIME_HOURS = (7, 20)           # Figures C–E: restrict plotted hours to 07:00–20:00

WORK_SCHEDULES = ("remote_worker", "onsite_worker")

RW_SHARE_LABELS = ["> 0.5", "0.2–0.5", "< 0.2"]
DURATION_LABELS = ["short", "medium", "long"]


In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 12, "axes.labelsize": 11,
                     "legend.frameon": True, "savefig.bbox": "tight"})

SCHEDULE_LABEL = {"remote_worker": "Remote workers", "onsite_worker": "On-site workers"}
SCHEDULE_COLOR = {"remote_worker": "#2b6cb0", "onsite_worker": "#dd6b20"}
TEL_AVIV_SEMEI = 5000  # CBS municipal code Tel Aviv-Yafo (not a statistical area)
TLV_ORDER = ["Tel Aviv-Yafo", "Rest of Gush Dan"]


def add_flag_tel_aviv(df, col="yishuv_stat_2022"):
    """flag_tel_aviv from yishuv_stat_2022 prefix (5000xxxx), not yishuv_stat == 5000."""
    y = df[col].astype("string").str.replace(r"\.0$", "", regex=True)
    flag = y.str.startswith(str(TEL_AVIV_SEMEI))
    df["flag_tel_aviv"] = flag.astype("boolean")
    df["tel_aviv_label"] = pd.Series(pd.NA, index=df.index, dtype="string")
    df.loc[flag.eq(True), "tel_aviv_label"] = TLV_ORDER[0]
    df.loc[flag.eq(False), "tel_aviv_label"] = TLV_ORDER[1]
    return df


def add_wilson_ci(df, count_col="k", n_col="n", z=1.96):
    """95% Wilson binomial confidence interval for a share k/n."""
    n = df[n_col].astype(float)
    k = df[count_col].astype(float)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return df.assign(share=p,
                     ci_low=(center - half).clip(lower=0),
                     ci_high=(center + half).clip(upper=1))


def share_by(df, group_cols, flag_col):
    """Share of rows where flag_col is True, per group, with Wilson CIs."""
    g = (df.groupby(group_cols, observed=True)
           .agg(n=(flag_col, "size"), k=(flag_col, "sum"))
           .reset_index())
    return add_wilson_ci(g)


def mean_by(df, group_cols, value_col):
    """Group mean with a 95% normal confidence interval."""
    g = (df.dropna(subset=[value_col])
           .groupby(group_cols, observed=True)[value_col]
           .agg(mean="mean", sd="std", n="size")
           .reset_index())
    se = g["sd"] / np.sqrt(g["n"])
    return g.assign(ci_low=g["mean"] - 1.96 * se, ci_high=g["mean"] + 1.96 * se)


def caption(text):
    print("\n" + textwrap.fill(" ".join(text.split()), 105))


TYPE_GROUP = {
    # Commercial: privately operated hospitality / retail
    "cafe": "Commercial",
    "coffee_shop": "Commercial",
    "bakery": "Commercial",
    "dessert_restaurant": "Commercial",
    "book_store": "Commercial",
    "bookstore": "Commercial",
    "tea_house": "Commercial",
    "breakfast_restaurant": "Commercial",
    "brunch_restaurant": "Commercial",
    "cat_cafe": "Commercial",
    "dog_cafe": "Commercial",
    # Civil: public / non-commercial cultural and community
    "library": "Civil",
    "museum": "Civil",
    "community_center": "Civil",
    "cultural_center": "Civil",
}
POI_GROUP_ORDER = ["Civil", "Commercial"]
PLACE_TYPE_ORDER = [True, False]
PLACE_TYPE_LABEL = {True: "Detected third places", False: "Other scored POIs"}
PLACE_TYPE_COLOR = {True: "#2b6cb0", False: "#dd6b20"}
MIN_HOUR_N = 30


def plot_hourly_share(
    df,
    series_col,
    series_order,
    series_label,
    series_color,
    ylabel,
    title,
    fig_id,
    flag_col=None,
    facet_row_col=None,
    facet_col_col="tel_aviv_label",
    facet_row_order=None,
    facet_col_order=None,
    min_hour=MIN_HOUR_N,
    hour_range=None,
    caption_text=None,
):
    """Hourly share of a binary visit flag, with Wilson bands.

    Pass visit-level rows + flag_col, or a pre-aggregated frame with columns
    n, k (and optionally share / ci_low / ci_high) already computed.
    hour_range=(lo, hi) keeps only hours lo..hi inclusive (e.g. daytime 7–20).
    """
    facet_col_order = list(facet_col_order or TLV_ORDER)
    if facet_row_col is None:
        facet_row_order = [None]
        nrows, ncols = 1, len(facet_col_order)
    else:
        facet_row_order = list(facet_row_order or POI_GROUP_ORDER)
        nrows, ncols = len(facet_row_order), len(facet_col_order)

    need = [series_col, "hour", facet_col_col]
    if facet_row_col is not None:
        need.append(facet_row_col)
    panel = df.dropna(subset=need).copy()
    panel = panel[panel[facet_col_col].isin(facet_col_order)]
    if facet_row_col is not None:
        panel = panel[panel[facet_row_col].isin(facet_row_order)]
    panel = panel[panel[series_col].isin(series_order)]
    if hour_range is not None:
        lo, hi = hour_range
        panel = panel[panel["hour"].between(lo, hi)]

    group_cols = [c for c in [facet_row_col, facet_col_col, series_col, "hour"] if c is not None]
    if flag_col is not None:
        agg = share_by(panel, group_cols, flag_col)
    else:
        if not {"n", "k"}.issubset(panel.columns):
            raise ValueError("Pre-aggregated input needs columns n and k (or pass flag_col)")
        agg = (
            panel.groupby(group_cols, observed=True)[["n", "k"]]
            .sum()
            .reset_index()
        )
        agg = add_wilson_ci(agg)
    agg = agg[agg["n"] >= min_hour]

    fig, axes = plt.subplots(
        nrows, ncols, figsize=(7.0 * ncols, 4.2 * nrows),
        sharex=True, sharey=True, squeeze=False,
    )
    for i, row_key in enumerate(facet_row_order):
        for j, col_key in enumerate(facet_col_order):
            ax = axes[i][j]
            sub = agg[agg[facet_col_col] == col_key]
            if facet_row_col is not None:
                sub = sub[sub[facet_row_col] == row_key]
            n_panel = int(sub.groupby(series_col)["n"].sum().sum()) if len(sub) else 0
            for series in series_order:
                d = sub[sub[series_col] == series].sort_values("hour")
                if d.empty:
                    continue
                color = series_color[series]
                label = series_label[series]
                ax.plot(d["hour"], d["share"] * 100, marker="o", ms=4, color=color, label=label)
                ax.fill_between(
                    d["hour"], d["ci_low"] * 100, d["ci_high"] * 100,
                    color=color, alpha=0.20,
                )
            ax.axvspan(EARLY_AFTERNOON_HOURS[0] - 0.5, EARLY_AFTERNOON_HOURS[1] + 0.5, color="grey", alpha=0.10, zorder=0)
            if facet_row_col is None:
                ax.set_title(f"{col_key} (n={n_panel:,})")
            else:
                ax.set_title(f"{row_key} · {col_key} (n={n_panel:,})")
            ax.set_xticks(
                range(hour_range[0], hour_range[1] + 1, 2) if hour_range else range(0, 24, 2)
            )
            if i == nrows - 1:
                ax.set_xlabel("Hour of arrival (local time)")
            if j == 0:
                ax.set_ylabel(ylabel)

    axes[0][0].legend(loc="upper left", fontsize=8)
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()
    if caption_text:
        caption(caption_text)
    return agg


def plot_hourly_mean(
    df,
    series_col,
    series_order,
    series_label,
    series_color,
    ylabel,
    title,
    fig_id,
    value_col=None,
    facet_row_col=None,
    facet_col_col="tel_aviv_label",
    facet_row_order=None,
    facet_col_order=None,
    min_hour=MIN_HOUR_N,
    hour_range=None,
    caption_text=None,
):
    """Hourly mean of a continuous visit metric, with 95% normal CI bands.

    Pass visit-level rows + value_col, or a pre-aggregated frame with columns
    mean, sd, n (optionally ci_low / ci_high) already at the panel grain.
    hour_range=(lo, hi) keeps only hours lo..hi inclusive (e.g. daytime 7–20).
    """
    facet_col_order = list(facet_col_order or TLV_ORDER)
    if facet_row_col is None:
        facet_row_order = [None]
        nrows, ncols = 1, len(facet_col_order)
    else:
        facet_row_order = list(facet_row_order or POI_GROUP_ORDER)
        nrows, ncols = len(facet_row_order), len(facet_col_order)

    need = [series_col, "hour", facet_col_col]
    if facet_row_col is not None:
        need.append(facet_row_col)
    panel = df.dropna(subset=need).copy()
    panel = panel[panel[facet_col_col].isin(facet_col_order)]
    if facet_row_col is not None:
        panel = panel[panel[facet_row_col].isin(facet_row_order)]
    panel = panel[panel[series_col].isin(series_order)]
    if hour_range is not None:
        lo, hi = hour_range
        panel = panel[panel["hour"].between(lo, hi)]

    group_cols = [c for c in [facet_row_col, facet_col_col, series_col, "hour"] if c is not None]
    if value_col is not None:
        agg = mean_by(panel, group_cols, value_col)
    else:
        if not {"mean", "sd", "n"}.issubset(panel.columns):
            raise ValueError("Pre-aggregated input needs columns mean, sd, n (or pass value_col)")
        # Already at (facet × series × hour); do not re-average means across cells.
        keep = group_cols + ["mean", "sd", "n"]
        for optional in ("ci_low", "ci_high"):
            if optional in panel.columns:
                keep.append(optional)
        agg = panel[keep].drop_duplicates(subset=group_cols).copy()
        if not {"ci_low", "ci_high"}.issubset(agg.columns):
            se = agg["sd"] / np.sqrt(agg["n"].astype(float))
            agg = agg.assign(ci_low=agg["mean"] - 1.96 * se, ci_high=agg["mean"] + 1.96 * se)
    agg = agg[agg["n"] >= min_hour]

    fig, axes = plt.subplots(
        nrows, ncols, figsize=(7.0 * ncols, 4.2 * nrows),
        sharex=True, sharey=True, squeeze=False,
    )
    for i, row_key in enumerate(facet_row_order):
        for j, col_key in enumerate(facet_col_order):
            ax = axes[i][j]
            sub = agg[agg[facet_col_col] == col_key]
            if facet_row_col is not None:
                sub = sub[sub[facet_row_col] == row_key]
            n_panel = int(sub.groupby(series_col)["n"].sum().sum()) if len(sub) else 0
            for series in series_order:
                d = sub[sub[series_col] == series].sort_values("hour")
                if d.empty:
                    continue
                color = series_color[series]
                label = series_label[series]
                ax.plot(d["hour"], d["mean"], marker="o", ms=4, color=color, label=label)
                ax.fill_between(
                    d["hour"], d["ci_low"], d["ci_high"],
                    color=color, alpha=0.20,
                )
            ax.axvspan(EARLY_AFTERNOON_HOURS[0] - 0.5, EARLY_AFTERNOON_HOURS[1] + 0.5, color="grey", alpha=0.10, zorder=0)
            if facet_row_col is None:
                ax.set_title(f"{col_key} (n={n_panel:,})")
            else:
                ax.set_title(f"{row_key} · {col_key} (n={n_panel:,})")
            ax.set_xticks(
                range(hour_range[0], hour_range[1] + 1, 2) if hour_range else range(0, 24, 2)
            )
            if i == nrows - 1:
                ax.set_xlabel("Hour of arrival (local time)")
            if j == 0:
                ax.set_ylabel(ylabel)

    axes[0][0].legend(loc="upper left", fontsize=8)
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()
    if caption_text:
        caption(caption_text)
    return agg


## 1. Place level — what was detected

One row per POI predicted as a third place (`prediction_third_place = 1`).

In [ ]:
df_places = utils.read_query_to_dataframe(f"""
SELECT * EXCEPT(geometry)
FROM `{PU_TABLE}`
WHERE prediction_third_place = 1
""")

df_places["primarytype"] = df_places["primarytype"].str.lower()
df_places["avg_dist_home_km"] = df_places["avg_dist_home"].astype(float) / 1000
df_places["avg_dist_home_rw_km"] = df_places["avg_dist_home_rw"].astype(float) / 1000
add_flag_tel_aviv(df_places)

print(f"Detected third places: {len(df_places):,}")
print(f"Tel Aviv-Yafo (semel_yishuv={TEL_AVIV_SEMEI}): "
      f"{int(df_places['flag_tel_aviv'].eq(True).sum()):,} | "
      f"Rest of Gush Dan: {int(df_places['flag_tel_aviv'].eq(False).sum()):,} | "
      f"missing SA: {int(df_places['flag_tel_aviv'].isna().sum()):,}")

## 2. All workers vs remote workers at the same places

`pois_visits_metrics_2022_2023` holds the same metrics computed twice per POI: once over all
visitors (`all_workers`) and once over remote workers only (`remote_workers`). Comparing the two
within the same place removes any place-composition effect — a paired test.

In [ ]:
df_metrics = utils.read_query_to_dataframe(f"""
SELECT m.* EXCEPT(geometry), p.primarytype, p.yishuv_stat_2022
FROM `{VISIT_METRICS_TABLE}` m
JOIN `{PU_TABLE}` p
  ON m.poi_id = p.id
WHERE p.prediction_third_place = 1
""")

df_metrics["share_long_visits_weekday"] = (df_metrics["n_long_visits_weekday"].astype(float)
                                           / df_metrics["n_visits_weekday"].astype(float))
df_metrics["share_visits_after_home"] = (df_metrics["n_visits_after_home_weekday"].astype(float)
                                         / df_metrics["n_visits_weekday"].astype(float))
df_metrics["avg_dist_home_km"] = df_metrics["avg_dist_home"].astype(float) / 1000
add_flag_tel_aviv(df_metrics)

# PU table still stores the all-days share; overwrite with the weekday share for place-level tables.
_share_map = (
    df_metrics.loc[df_metrics["worker_type"].eq("all_workers")]
    .set_index("poi_id")["share_visits_after_home"]
)
df_places["share_visits_after_home"] = df_places["id"].map(_share_map)

METRIC_COLS = ["avg_duration_hours_weekday", "share_long_visits_weekday", "avg_dist_home_km",
               "share_dwells_home_lt_1km", "share_visits_after_home",
               "share_morning", "share_early_afternoon", "share_late_afternoon", "share_evening",
               "share_weekday", "visit_frequency"]

by_worker = (df_metrics.groupby("worker_type")[METRIC_COLS].mean().T.round(3))
by_worker["difference"] = (by_worker["remote_workers"] - by_worker["all_workers"]).round(3)

In [ ]:
# Paired comparison within place: remote workers vs all visitors of the same POI
paired = (df_metrics.pivot_table(index="poi_id", columns="worker_type", values=METRIC_COLS))

rows = []
for col in METRIC_COLS:
    pair = paired[col].dropna()
    if len(pair) < 20:
        continue
    stat, pval = stats.wilcoxon(pair["remote_workers"], pair["all_workers"])
    rows.append({"metric": col,
                 "n_places": len(pair),
                 "all_workers": pair["all_workers"].mean(),
                 "remote_workers": pair["remote_workers"].mean(),
                 "difference": pair["remote_workers"].mean() - pair["all_workers"].mean(),
                 "p_value": pval})

pd.DataFrame(rows).round(4).sort_values("difference", ascending=False)

## 3. Location context

Two contextual attributes per place:

* `flag_tel_aviv` — whether the place sits in Tel Aviv-Yafo (`semel_yishuv` = 5000, from the
  prefix of `yishuv_stat_2022`). Contrast is Tel Aviv-Yafo versus the rest of Gush Dan.
* `n_same_primarytype_1km` — how many other Google POIs of the **same category** lie within 1 km.
  Computed here against the full Google POI universe (`google_3rdplaces_cities`), so it measures
  local supply of the same amenity type, not just other detected third places.

The figure below holds the Tel Aviv flag fixed and compares **detected third places** with the
remaining scored POIs on place-level features. Features are z-scored **inside each location
group**, so the dashed line is that group's own mean and sits between the two series (closer to
the larger series). Mixed units share one axis.

```text
Metric: dist_home_km, share_home_lt_1km, duration_weekday_h,
        hourly_signal_std_percentile_by_keyword,
        share_morning_weekday, share_early_afternoon_weekday,
        share_evening_weekday, share_early_afternoon_long_weekday
        (z-scored within Tel Aviv / rest of Gush Dan)
Categories: detected third places vs remaining scored POIs
X: group mean of the within-location z-score, 95% normal CI
Facets: one panel per tel_aviv_label
```

In [ ]:
df_density = utils.read_query_to_dataframe(f"""
WITH tp AS (
  SELECT id, LOWER(primarytype) AS primarytype, geometry
  FROM `{PU_TABLE}`
  WHERE prediction_third_place = 1
)
SELECT
  tp.id,
  COUNTIF(u.id IS NOT NULL AND u.id != tp.id) AS n_same_primarytype_1km
FROM tp
LEFT JOIN `{POI_UNIVERSE_TABLE}` u
  ON LOWER(u.primarytype) = tp.primarytype
 AND ST_DWITHIN(tp.geometry, u.geometry, {DENSITY_RADIUS_M})
GROUP BY tp.id
""")

df_places = df_places.merge(df_density, on="id", how="left")
df_places["n_same_primarytype_1km"] = df_places["n_same_primarytype_1km"].astype(float)

df_places["density_tertile"] = pd.qcut(
    df_places["n_same_primarytype_1km"].rank(method="first"), 3,
    labels=["Low density", "Medium density", "High density"])

df_places["n_same_primarytype_1km"].describe().round(2)

In [ ]:
by_neigh = (df_places
            .groupby("tel_aviv_label")
            .agg(n_places=("id", "size"),
                 same_category_1km=("n_same_primarytype_1km", "median"),
                 duration_weekday_h=("avg_duration_hours_weekday", "mean"),
                 duration_weekday_rw_h=("avg_duration_hours_weekday_rw", "mean"),
                 dist_home_km=("avg_dist_home_km", "mean"),
                 share_home_lt_1km=("share_dwells_home_lt_1km", "mean"),
                 share_after_home=("share_visits_after_home", "mean"),
                 rw_visit_share=("remote_workers_visits_share_weekday", "mean"))
            .round(3))

by_density = (df_places
              .groupby("density_tertile", observed=True)
              .agg(n_places=("id", "size"),
                   same_category_1km=("n_same_primarytype_1km", "median"),
                   duration_weekday_h=("avg_duration_hours_weekday", "mean"),
                   dist_home_km=("avg_dist_home_km", "mean"),
                   share_after_home=("share_visits_after_home", "mean"),
                   rw_visit_share=("remote_workers_visits_share_weekday", "mean"))
              .round(3))

In [ ]:
# Metric: place-level features, z-scored within Tel Aviv vs rest of Gush Dan
# Categories: detected third places vs remaining scored POIs
# X: group mean of the within-location z-score, with 95% normal CI
# Facets: one panel per tel_aviv_label

df_places_all = utils.read_query_to_dataframe(f"""
WITH wd AS (
  SELECT
    d.id AS poi_id,
    SAFE_DIVIDE(
      COUNTIF(EXTRACT(HOUR FROM d.start_date_time)
              BETWEEN {MORNING_HOURS[0]} AND {MORNING_HOURS[1]}),
      COUNT(*)) AS share_morning_weekday,
    SAFE_DIVIDE(
      COUNTIF(EXTRACT(HOUR FROM d.start_date_time)
              BETWEEN {EARLY_AFTERNOON_HOURS[0]} AND {EARLY_AFTERNOON_HOURS[1]}),
      COUNT(*)) AS share_early_afternoon_weekday,
    SAFE_DIVIDE(
      COUNTIF(EXTRACT(HOUR FROM d.start_date_time)
              BETWEEN {LATE_AFTERNOON_HOURS[0]} AND {LATE_AFTERNOON_HOURS[1]}),
      COUNT(*)) AS share_late_afternoon_weekday,
    SAFE_DIVIDE(
      COUNTIF(EXTRACT(HOUR FROM d.start_date_time)
              BETWEEN {EVENING_HOURS[0]} AND {EVENING_HOURS[1]}),
      COUNT(*)) AS share_evening_weekday,
    SAFE_DIVIDE(
      COUNTIF(EXTRACT(HOUR FROM d.start_date_time)
              BETWEEN {EARLY_AFTERNOON_HOURS[0]} AND {EARLY_AFTERNOON_HOURS[1]}
              AND d.duration_hours > {LONG_VISIT_HOURS}),
      COUNT(*)) AS share_early_afternoon_long_weekday
  FROM `{DWELLS_TABLE}` d
  JOIN `{PU_TABLE}` p ON d.id = p.id
  WHERE NOT d.flag_home
    AND NOT d.flag_work
    AND d.day_type = 'weekday'
    AND d.month BETWEEN '{START_PERIOD}' AND '{END_PERIOD}'
  GROUP BY d.id
)
SELECT
  p.id,
  p.prediction_third_place,
  p.yishuv_stat_2022,
  p.avg_dist_home,
  p.share_dwells_home_lt_1km,
  p.avg_duration_hours_weekday,
  p.hourly_signal_std_percentile_by_keyword,
  p.share_visits_after_work,
  p.avg_dist_home_work_ratio,
  m.avg_dist_work,
  SAFE_DIVIDE(m.n_visits_after_home_weekday, m.n_visits_weekday) AS share_visits_after_home_weekday,
  wd.share_morning_weekday,
  wd.share_early_afternoon_weekday,
  wd.share_late_afternoon_weekday,
  wd.share_evening_weekday,
  wd.share_early_afternoon_long_weekday
FROM `{PU_TABLE}` p
LEFT JOIN `{VISIT_METRICS_TABLE}` m
  ON m.poi_id = p.id
 AND m.worker_type = 'all_workers'
LEFT JOIN wd
  ON wd.poi_id = p.id
""")

df_places_all["dist_home_km"] = df_places_all["avg_dist_home"].astype(float) / 1000
df_places_all["share_home_lt_1km"] = df_places_all["share_dwells_home_lt_1km"].astype(float)
df_places_all["duration_weekday_h"] = df_places_all["avg_duration_hours_weekday"].astype(float)
df_places_all["share_from_home"] = df_places_all["share_visits_after_home_weekday"].astype(float)
df_places_all["share_from_work"] = df_places_all["share_visits_after_work"].astype(float)
dist_work_m = pd.to_numeric(df_places_all["avg_dist_work"], errors="coerce")
ratio = pd.to_numeric(df_places_all["avg_dist_home_work_ratio"], errors="coerce")
home_m = pd.to_numeric(df_places_all["avg_dist_home"], errors="coerce")
recovered_m = np.where(ratio.gt(0), home_m / ratio, np.nan)
df_places_all["dist_work_km"] = np.where(dist_work_m.notna(), dist_work_m, recovered_m) / 1000
df_places_all["hourly_signal_std_percentile_by_keyword"] = (
    df_places_all["hourly_signal_std_percentile_by_keyword"].astype(float)
)
for col in ("share_morning_weekday", "share_early_afternoon_weekday",
            "share_late_afternoon_weekday", "share_evening_weekday",
            "share_early_afternoon_long_weekday"):
    df_places_all[col] = df_places_all[col].astype(float)
df_places_all["is_detected"] = df_places_all["prediction_third_place"].astype(int).eq(1)
add_flag_tel_aviv(df_places_all)

FEATURE_ORDER = [
    "dist_home_km",
    # "dist_work_km",
    # "share_home_lt_1km",
    "share_from_home",
    "share_from_work",
    "duration_weekday_h",
    # "hourly_signal_std_percentile_by_keyword",
    "share_morning_weekday",
    # "share_early_afternoon_weekday",
    # "share_late_afternoon_weekday",
    # "share_evening_weekday",
    "share_early_afternoon_long_weekday",
]
FEATURE_LABEL = {
    "dist_home_km": "Distance from home (km)",
    # "dist_work_km": "Distance from work (km)",
    # "share_home_lt_1km": "Share of local visits (<1 km from home)",
    "share_from_home": "Share of visits arriving from home",
    "share_from_work": "Share of visits arriving from work",
    "duration_weekday_h": "Visit duration (h)",
    # "hourly_signal_std_percentile_by_keyword": "Hourly rhythm variability (pctl. by keyword)",
    "share_morning_weekday": f"Share of visits, morning ({MORNING_HOURS[0]}–{MORNING_HOURS[1]})",
    "share_early_afternoon_weekday": f"Share of visits, early afternoon ({EARLY_AFTERNOON_HOURS[0]}–{EARLY_AFTERNOON_HOURS[1]})",
    "share_late_afternoon_weekday": (
        f"Share of weekday visits, late afternoon "
        f"({LATE_AFTERNOON_HOURS[0]}–{LATE_AFTERNOON_HOURS[1]})"
    ),
    "share_evening_weekday": f"Share of visits, evening ({EVENING_HOURS[0]}–{EVENING_HOURS[1]})",
    "share_early_afternoon_long_weekday": (
        f"Share of visits that are long early-afternoon "
        f"(>{LONG_VISIT_HOURS:g} h, {EARLY_AFTERNOON_HOURS[0]}–{EARLY_AFTERNOON_HOURS[1]})"
    ),
}
GROUP_LABEL = {True: "Detected third places", False: "Other scored POIs"}
GROUP_COLOR = {True: "#2b6cb0", False: "#dd6b20"}
MIN_CELL_PLACES = 20

panel = df_places_all.dropna(subset=["tel_aviv_label"] + FEATURE_ORDER).copy()
panel = panel[panel["tel_aviv_label"].isin(TLV_ORDER)]

print(
    "Weekday visit behaviour of detected third places vs other scored POIs, split by Tel Aviv-Yafo vs rest of Gush Dan\n"
    "Metric: place-level features (z-scored within Tel Aviv / rest of Gush Dan)\n"
    "Categories: detected third places vs remaining scored POIs\n"
    "X: standardized mean with 95% normal CI\n"
    "Facets: tel_aviv_label\n"
)


def zscore_within(s):
    sd = s.std(ddof=1)
    if sd == 0 or pd.isna(sd):
        return s * 0.0
    return (s - s.mean()) / sd


# Standardize independently inside each location group. 0 = that group's mean.
for col in FEATURE_ORDER:
    panel[col + "_z"] = panel.groupby("tel_aviv_label")[col].transform(zscore_within)

rows = []
for feat in FEATURE_ORDER:
    g = mean_by(panel, ["tel_aviv_label", "is_detected"], feat + "_z")
    g["feature"] = feat
    rows.append(g)
z_means = pd.concat(rows, ignore_index=True)
z_means = z_means[z_means["n"] >= MIN_CELL_PLACES]

n_table = (
    panel.groupby(["tel_aviv_label", "is_detected"], observed=True)
         .size()
         .unstack("is_detected", fill_value=0)
         .reindex(TLV_ORDER)
         .rename(columns={False: "n_other", True: "n_detected"})
)
work_by = (
    panel.groupby(["tel_aviv_label", "is_detected"], observed=True)
         .agg(dist_work_km=("dist_work_km", "mean"),
              share_from_home=("share_from_home", "mean"),
              share_from_work=("share_from_work", "mean"))
)
for col in ("dist_work_km", "share_from_home", "share_from_work"):
    wide = (work_by[col].unstack("is_detected")
            .reindex(TLV_ORDER)
            .rename(columns={False: f"{col}_other", True: f"{col}_detected"}))
    n_table = n_table.join(wide)


fig, axes = plt.subplots(1, 2, figsize=(12, 8.5), sharex=True, sharey=True)
y = np.arange(len(FEATURE_ORDER))
offset = {True: -0.14, False: 0.14}

for ax, loc in zip(axes, TLV_ORDER):
    d_loc = z_means[z_means["tel_aviv_label"] == loc]
    n_det = int(n_table.loc[loc, "n_detected"])
    n_oth = int(n_table.loc[loc, "n_other"])
    for detected in (True, False):
        d = (d_loc[d_loc["is_detected"] == detected]
             .set_index("feature")
             .reindex(FEATURE_ORDER))
        if d["mean"].isna().all():
            continue
        ax.errorbar(
            d["mean"], y + offset[detected],
            xerr=[d["mean"] - d["ci_low"], d["ci_high"] - d["mean"]],
            fmt="o", ms=6,
            color=GROUP_COLOR[detected], ecolor=GROUP_COLOR[detected],
            elinewidth=1.4, capsize=2.5, zorder=3,
            label=f"{GROUP_LABEL[detected]} (n={n_det if detected else n_oth:,})",
        )
    ax.axvline(0, ls="--", lw=1.2, color="#718096", zorder=1,
               **({"label": "Location-group mean (z = 0)"} if ax is axes[0] else {}))
    ax.set_yticks(y)
    ax.set_yticklabels([FEATURE_LABEL[f] for f in FEATURE_ORDER])
    ax.set_ylim(len(FEATURE_ORDER) - 0.5, -0.5)
    ax.set_title(loc)
    ax.grid(axis="y", visible=False)
    ax.legend(fontsize=8, loc="best")
    ax.set_xlabel("Standardized mean within location group (z-score)")
fig.suptitle("Detected third places vs other scored POIs, Tel Aviv-Yafo vs rest of Gush Dan",
             y=0.995)
plt.tight_layout()
plt.show()

caption(f"""**Fig. N.** Place-level weekday visit behaviour of PU-detected third places versus the remaining
scored POIs, split by Tel Aviv-Yafo versus the rest of Gush Dan (`semel_yishuv` = 5000).
Features are z-scored independently inside each location group, so the dashed line is that
group's own mean and lies between the two series. Points are group means; error bars are 95%
normal confidence intervals (mean ± 1.96 · sd/√n). *Notes:* unit of analysis is the place;
weekday timing shares are among weekday visits 2022-2023 (morning {MORNING_HOURS[0]}–{MORNING_HOURS[1]},
early afternoon {EARLY_AFTERNOON_HOURS[0]}–{EARLY_AFTERNOON_HOURS[1]}, late afternoon {LATE_AFTERNOON_HOURS[0]}–{LATE_AFTERNOON_HOURS[1]},
evening {EVENING_HOURS[0]}–{EVENING_HOURS[1]}; long early-afternoon is duration > {LONG_VISIT_HOURS:g} h arriving
in the early-afternoon window); location cells with fewer than 20 places are omitted.""")


## 4. Visit-level panel

Every dwell at a detected third place between 2022-01 and 2023-12, joined to the visitor's monthly
profile. Home and work dwells are excluded, and only users labelled `remote_worker` or
`onsite_worker` are kept (users whose schedule could not be determined are dropped).

In [ ]:
df_visits = utils.read_query_to_dataframe(f"""
WITH tp AS (
  SELECT id, LOWER(primarytype) AS primarytype, yishuv_stat_2022
  FROM `{PU_TABLE}`
  WHERE prediction_third_place = 1
),
-- Monthly amenity affinity, rebuilt from non-third-place dwells.
-- Visits to the detected third places themselves are excluded so the affinity flags stay
-- independent of the behaviour they are used to explain.
affinity AS (
  SELECT
    a.identifier,
    a.month,
    COUNTIF(a.pois_category IN ('food', 'cafe', 'coffee_shop', 'bakery', 'tea_house')) > 1
      AS like_cafes,
    COUNTIF(a.pois_category = 'retail') > 1 AS like_shopping,
    COUNTIF(a.pois_category IN ('parks', 'outdoor_attractions')) > 1 AS like_outdoor
  FROM `{DWELLS_TABLE}` a
  LEFT JOIN tp ON a.id = tp.id
  WHERE NOT a.flag_home
    AND NOT a.flag_work
    AND tp.id IS NULL
    AND a.month BETWEEN '{START_PERIOD}' AND '{END_PERIOD}'
  GROUP BY a.identifier, a.month
)
SELECT
  d.id AS poi_id,
  d.identifier,
  d.month,
  EXTRACT(HOUR FROM d.start_date_time) AS hour,
  d.day_type,
  d.duration_hours,
  d.dist_to_home,
  (d.prev_flag_home AND d.time_diff_from_prev <= {FROM_PREV_MAX_GAP_H}) AS from_home,
  (d.prev_flag_work AND d.time_diff_from_prev <= {FROM_PREV_MAX_GAP_H}) AS from_work,
  tp.primarytype,
  tp.yishuv_stat_2022,
  -- work / home context
  up.work_schedule,
  up.flag_remote_work_smoothed_max_thresh AS rw_share,
  up.work_context_type,
  up.work_neighborhood_type,
  up.home_neighborhood_type,
  up.employment_zone,
  -- census-based home-area tags
  up.home_yishuv_stat_2022,
  up.home_tag_ses,
  up.home_tag_lifecycle,
  up.home_tag_religion,
  up.home_tag_education,
  up.home_tag_mobility,
  up.home_tag_occupation_proxy,
  -- behavioural flags from the profile table
  up.FLAG_STUDENT,
  up.FLAG_MEDICAL,
  up.FLAG_PARENT,
  up.FLAG_RELIGIOUS,
  up.FLAG_LIKE_ACCOMMODATION,
  up.flag_reliable_min_5days,
  -- affinity flags rebuilt here
  COALESCE(af.like_cafes, FALSE) AS like_cafes,
  COALESCE(af.like_shopping, FALSE) AS like_shopping,
  COALESCE(af.like_outdoor, FALSE) AS like_outdoor
FROM `{DWELLS_TABLE}` d
JOIN tp
  ON d.id = tp.id
JOIN `{USER_PROFILES_TABLE}` up
  ON d.identifier = up.identifier
 AND d.month = up.month
LEFT JOIN affinity af
  ON d.identifier = af.identifier
 AND d.month = af.month
WHERE NOT d.flag_home
  AND NOT d.flag_work
  AND d.month BETWEEN '{START_PERIOD}' AND '{END_PERIOD}'
  AND up.work_schedule IN {WORK_SCHEDULES}
""")

In [ ]:
df_visits["hour"] = df_visits["hour"].astype(int)
df_visits["duration_hours"] = df_visits["duration_hours"].astype(float)
df_visits["dist_home_km"] = df_visits["dist_to_home"].astype(float) / 1000
df_visits["from_home"] = df_visits["from_home"].fillna(False).astype(bool)
df_visits["from_work"] = df_visits["from_work"].fillna(False).astype(bool)

df_visits["rw_share"] = df_visits["rw_share"].astype(float)
df_visits["is_long"] = df_visits["duration_hours"] > LONG_VISIT_HOURS
df_visits["is_weekday"] = df_visits["day_type"] == "weekday"
df_visits["is_morning"] = df_visits["hour"].between(*MORNING_HOURS)
df_visits["is_early_afternoon"] = df_visits["hour"].between(*EARLY_AFTERNOON_HOURS)
df_visits["is_late_afternoon"] = df_visits["hour"].between(*LATE_AFTERNOON_HOURS)
df_visits["is_evening"] = df_visits["hour"].between(*EVENING_HOURS)
df_visits["near_home"] = df_visits["dist_home_km"] < 1
df_visits["schedule_label"] = df_visits["work_schedule"].map(SCHEDULE_LABEL)

# Mutually exclusive RW intensity (user-month) and visit duration bins
# Groups: >0.5 | [0.2, 0.5] | <0.2. Duration: short <10min | medium <=1h | long >1h
df_visits["rw_share_group"] = np.select(
    [df_visits["rw_share"] > 0.5,
     df_visits["rw_share"] >= 0.2],
    ["> 0.5", "0.2–0.5"],
    default="< 0.2",
)
df_visits.loc[df_visits["rw_share"].isna(), "rw_share_group"] = np.nan
df_visits["rw_share_group"] = pd.Categorical(
    df_visits["rw_share_group"], categories=RW_SHARE_LABELS, ordered=True)

df_visits["duration_bin"] = np.select(
    [df_visits["duration_hours"] < SHORT_VISIT_HOURS,
     df_visits["duration_hours"] <= LONG_VISIT_HOURS],
    ["short", "medium"],
    default="long",
)
df_visits["duration_bin"] = pd.Categorical(
    df_visits["duration_bin"], categories=DURATION_LABELS, ordered=True)

# profile flags arrive as 0/1 integers or nullable booleans; normalise them all to plain bool
PROFILE_FLAGS = ["FLAG_STUDENT", "FLAG_MEDICAL", "FLAG_PARENT", "FLAG_RELIGIOUS",
                 "FLAG_LIKE_ACCOMMODATION",
                 "flag_reliable_min_5days", "employment_zone",
                 "like_cafes", "like_shopping", "like_outdoor"]
for col in PROFILE_FLAGS:
    df_visits[col] = df_visits[col].fillna(0).astype(float).astype(bool)

df_visits = df_visits.merge(
    df_places[["id", "n_same_primarytype_1km", "density_tertile",
               "flag_tel_aviv", "tel_aviv_label"]].rename(columns={"id": "poi_id"}),
    on="poi_id", how="left")

df_weekday = df_visits[df_visits["is_weekday"]].copy()
df_weekend = df_visits[~df_visits["is_weekday"]].copy()

# Baseline for reading every "remote workers' share of visits" figure below: the panel keeps only
# users with a determined schedule, so the denominator is remote + on-site workers, not all visitors.
RW_BASELINE = df_weekday["work_schedule"].eq("remote_worker").mean()
print(f"Remote workers make {RW_BASELINE:.1%} of classified weekday visits (panel baseline); "
      f"they are {df_places['remote_workers_visits_share_weekday'].mean():.1%} of all weekday "
      f"visits at the average detected place.")

(df_visits.groupby(["work_schedule", "day_type"])
          .agg(visits=("poi_id", "size"), users=("identifier", "nunique"))
          .unstack("day_type"))


In [ ]:

CATEGORY_MAP = {
    "bakery": "bakery",
    "cafe": "cafe",
    "coffee_shop": "coffee_shop",
    "dessert_restaurant": "cafe",
    "tea_house": "cafe",
    "breakfast_restaurant": "cafe",
    "museum": "museum",
    "library": "library",
    "community_center": "community_center",
    "cultural_center": "cultural_center",
    "book_store": "book_store",
    "bookstore": "book_store",
}

In [ ]:
df_weekday['primarytype_agg'] = df_weekday.primarytype.map(CATEGORY_MAP)

### Visit duration by remote-work intensity

Metric: share of visits in each duration bin (short / medium / long).  
Categories: user-month `rw_share` groups (`flag_remote_work_smoothed_max_thresh`).  
X: share of visits (%) with 95% Wilson CIs.

- **short** — duration < 10 minutes  
- **medium** — 10 minutes ≤ duration ≤ 1 hour  
- **long** — duration > 1 hour  
- **RW groups** — `> 0.5`, `0.2–0.5`, `< 0.2`


In [ ]:
# Pivot: % of visits by duration bin within each RW-share group (weekday panel)
dur_panel = df_weekday[df_weekday['FLAG_PARENT']==False].dropna(subset=["rw_share_group", "duration_bin"]).copy()

counts = (
    dur_panel.groupby(["rw_share_group", "duration_bin"], observed=True)
             .size()
             .rename("n")
             .reset_index()
)
totals = counts.groupby("rw_share_group", observed=True)["n"].transform("sum")
counts["share"] = counts["n"] / totals

pivot_pct = (
    counts.pivot(index="rw_share_group", columns="duration_bin", values="share")
          .reindex(index=RW_SHARE_LABELS, columns=DURATION_LABELS)
          * 100
).round(1)
pivot_n = (
    counts.pivot(index="rw_share_group", columns="duration_bin", values="n")
          .reindex(index=RW_SHARE_LABELS, columns=DURATION_LABELS)
)
pivot_n["n_visits"] = pivot_n.sum(axis=1)
n_users = (
    dur_panel.groupby("rw_share_group", observed=True)["identifier"]
             .nunique()
             .reindex(RW_SHARE_LABELS)
)
pivot_n["avg_frequency"] = (pivot_n["n_visits"] / n_users).round(2)

print("Visit counts by RW share × duration (weekday); avg_frequency = weekday visits per unique user")
display(
    pivot_n.assign(**{c: pivot_n[c].astype("Int64") for c in list(DURATION_LABELS) + ["n_visits"]})
)
print("\nRow % (duration mix within RW group)")
display(pivot_pct.assign(total=pivot_pct.sum(axis=1).round(1)))

# Wilson CIs for each (group × duration) share
ci_rows = []
for g in RW_SHARE_LABELS:
    sub = dur_panel[dur_panel["rw_share_group"] == g]
    n = len(sub)
    for dur in DURATION_LABELS:
        k = int((sub["duration_bin"] == dur).sum())
        row = add_wilson_ci(pd.DataFrame([{"n": n, "k": k}])).iloc[0]
        ci_rows.append({
            "rw_share_group": g,
            "duration_bin": dur,
            "n": n,
            "k": k,
            "share": row["share"],
            "ci_low": row["ci_low"],
            "ci_high": row["ci_high"],
        })
dur_ci = pd.DataFrame(ci_rows)

MIN_CELL = 200
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.8), sharey=True)
colors = {"short": "#a0aec0", "medium": "#3182ce", "long": "#c05621"}
for ax, dur in zip(axes, DURATION_LABELS):
    d = (dur_ci[dur_ci["duration_bin"] == dur]
         .set_index("rw_share_group")
         .reindex(RW_SHARE_LABELS))
    d = d[d["n"] >= MIN_CELL]
    overall = (dur_panel["duration_bin"] == dur).mean() * 100
    y = np.arange(len(d))
    ax.errorbar(
        d["share"] * 100, y,
        xerr=[(d["share"] - d["ci_low"]) * 100, (d["ci_high"] - d["share"]) * 100],
        fmt="o", ms=6, color=colors[dur], ecolor=colors[dur],
        elinewidth=1.4, capsize=2.5,
    )
    ax.axvline(overall, ls="--", lw=1.2, color="#718096",
               label=f"All visits ({overall:.1f}%)")
    ax.set_yticks(y)
    ax.set_yticklabels([f"rw_share {g}" for g in d.index])
    ax.set_ylim(len(d) - 0.5, -0.5)
    ax.set_xlabel("Share of visits (%)")
    ax.set_title({"short": "Short (<10 min)",
                  "medium": "Medium (10 min–1 h)",
                  "long": "Long (>1 h)"}[dur])
    ax.legend(fontsize=8, loc="best")
fig.suptitle("Visit duration mix by remote-work intensity (weekday third-place visits)",
             y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

print(
    "Fig. Duration × RW share. Share of weekday visits in each duration bin by user-month "
    "remote-work intensity (flag_remote_work_smoothed_max_thresh). Points show percentages; "
    "error bars are 95% Wilson binomial CIs. Dashed line = overall sample share. "
    "Unit of analysis: visits (frequent visitors weigh more). "
    f"Bins: short <{SHORT_VISIT_HOURS*60:.0f} min; medium ≤{LONG_VISIT_HOURS:g} h; "
    f"long >{LONG_VISIT_HOURS:g} h. Groups with n < {MIN_CELL} omitted from the plot."
)


## 5. Figures

### Figures A and B — When each type of worker visits third places

Fig. A2 repeats the hourly layout of Fig. A, splitting by **place type** (detected third places vs other scored POIs) instead of visitor work schedule.

Figures **C–E** reuse one hourly-share function for three visit flags (near home, long visit, from home). Each flag is shown for remote vs on-site workers at third places, and for third places vs other scored POIs, both as Tel Aviv vs rest of Gush Dan and as civil/commercial × location.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), sharey=True)

for ax, data, period in zip(axes, [df_weekday, df_weekend],
                            ["Weekdays (Sun-Thu)", "Weekend (Fri-Sat)"]):
    hourly = share_by(data.assign(one=True), ["work_schedule", "hour"], "one")
    # share of each group's visits falling in that hour
    totals = data.groupby("work_schedule").size()
    hourly["n"] = hourly["work_schedule"].map(totals)
    hourly = add_wilson_ci(hourly)

    width = 0.4
    for i, sched in enumerate(WORK_SCHEDULES):
        d = hourly[hourly["work_schedule"] == sched].sort_values("hour")
        x = d["hour"] + (i - 0.5) * width
        ax.bar(x, d["share"] * 100, width=width, label=SCHEDULE_LABEL[sched],
               color=SCHEDULE_COLOR[sched], alpha=0.9)
        ax.errorbar(x, d["share"] * 100,
                    yerr=[(d["share"] - d["ci_low"]) * 100, (d["ci_high"] - d["share"]) * 100],
                    fmt="none", ecolor="0.3", elinewidth=0.7, capsize=1.5)

    ax.axvspan(EARLY_AFTERNOON_HOURS[0] - 0.5, EARLY_AFTERNOON_HOURS[1] + 0.5, color="grey", alpha=0.10, zorder=0)
    ax.set_title(period)
    ax.set_xlabel("Hour of arrival (local time)")
    ax.set_xticks(range(0, 24, 2))

axes[0].set_ylabel("Share of the group's visits (%)")
axes[0].legend(title="Visitor type", loc="upper left")
fig.suptitle("Hourly rhythm of visits to detected third places, remote vs on-site workers, "
             "Gush Dan 2022-2023", y=1.02)
plt.show()

caption("""**Fig. A.** Hourly distribution of visits to PU-detected third places by visitor work
schedule, weekdays and weekend, 2022-2023. Remote workers shift a larger share of their visits into
the working-day window (shaded, 10:00-15:00), whereas on-site workers concentrate in the evening
and at weekends. Source: Habidatum GPS dwells joined to monthly user profiles. *Notes:* visits at
550 detected third places; home and work dwells excluded; error bars are 95% Wilson binomial
confidence intervals.""")

In [ ]:
# Metric: share of the group's visits that arrive in this hour
# Categories: detected third places vs other scored POIs (not third)
# X: hour of arrival (local time)
# Facets: weekdays vs weekend

df_hourly_place = utils.read_query_to_dataframe(f"""
SELECT
  p.prediction_third_place = 1 AS is_third,
  EXTRACT(HOUR FROM d.start_date_time) AS hour,
  d.day_type = 'weekday' AS is_weekday,
  COUNT(*) AS k
FROM `{DWELLS_TABLE}` d
JOIN `{PU_TABLE}` p
  ON d.id = p.id
JOIN `{USER_PROFILES_TABLE}` up
  ON d.identifier = up.identifier
 AND d.month = up.month
WHERE NOT d.flag_home
  AND NOT d.flag_work
  AND d.month BETWEEN '{START_PERIOD}' AND '{END_PERIOD}'
  AND up.work_schedule IN {WORK_SCHEDULES}
GROUP BY 1, 2, 3
""")

PLACE_TYPE_ORDER = (True, False)
PLACE_TYPE_LABEL = {True: "Detected third places", False: "Other scored POIs"}
PLACE_TYPE_COLOR = {True: "#2b6cb0", False: "#dd6b20"}

n_by_type = (df_hourly_place.groupby(["is_third", "is_weekday"], as_index=False)["k"]
             .sum().rename(columns={"k": "n"}))
print(n_by_type.assign(label=n_by_type["is_third"].map(PLACE_TYPE_LABEL)).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), sharey=True)

for ax, is_wd, period in zip(axes, [True, False],
                             ["Weekdays (Sun-Thu)", "Weekend (Fri-Sat)"]):
    data = df_hourly_place[df_hourly_place["is_weekday"] == is_wd].copy()
    totals = data.groupby("is_third")["k"].sum()
    hourly = data.copy()
    hourly["n"] = hourly["is_third"].map(totals)
    hourly = add_wilson_ci(hourly)

    width = 0.4
    for i, is_third in enumerate(PLACE_TYPE_ORDER):
        d = hourly[hourly["is_third"] == is_third].sort_values("hour")
        x = d["hour"] + (i - 0.5) * width
        ax.bar(x, d["share"] * 100, width=width, label=PLACE_TYPE_LABEL[is_third],
               color=PLACE_TYPE_COLOR[is_third], alpha=0.9)
        ax.errorbar(x, d["share"] * 100,
                    yerr=[(d["share"] - d["ci_low"]) * 100, (d["ci_high"] - d["share"]) * 100],
                    fmt="none", ecolor="0.3", elinewidth=0.7, capsize=1.5)

    ax.axvspan(EARLY_AFTERNOON_HOURS[0] - 0.5, EARLY_AFTERNOON_HOURS[1] + 0.5, color="grey", alpha=0.10, zorder=0)
    ax.set_title(period)
    ax.set_xlabel("Hour of arrival (local time)")
    ax.set_xticks(range(0, 24, 2))

axes[0].set_ylabel("Share of the group's visits (%)")
axes[0].legend(title="Place type", loc="upper left")
fig.suptitle("Hourly rhythm of visits to detected third places vs other scored POIs, "
             "Gush Dan 2022-2023", y=1.02)
plt.show()

caption("""**Fig. A2.** Hourly distribution of visits by place type, weekdays and weekend, 2022-2023.
Detected third places (blue) are compared with other PU-scored POIs that were not classified as
third places (orange). Shares are within each place-type group, so the two series each sum to 100%
over the day. Source: Habidatum GPS dwells joined to monthly user profiles. *Notes:* same visitor
sample as Fig. A (remote and on-site workers); home and work dwells excluded; error bars are 95%
Wilson binomial confidence intervals; shaded window is 10:00-15:00.""")


In [ ]:
rw_weekday = df_weekday[(df_weekday["work_schedule"] == "remote_worker")
                        & df_weekday["home_tag_ses"].notna()]

hourly_ses = share_by(rw_weekday.assign(one=True), ["home_tag_ses", "hour"], "one")
totals_ses = rw_weekday.groupby("home_tag_ses").size()
hourly_ses["n"] = hourly_ses["home_tag_ses"].map(totals_ses)
hourly_ses = add_wilson_ci(hourly_ses)

SES_ORDER = ["USR_SES_LOW", "USR_SES_MED", "USR_SES_HIGH"]
SES_LABEL = {"USR_SES_LOW": "Low SES home area", "USR_SES_MED": "Medium SES home area",
             "USR_SES_HIGH": "High SES home area"}

fig, ax = plt.subplots(figsize=(12, 4.4))
width = 0.27
for i, ses in enumerate(SES_ORDER):
    d = hourly_ses[hourly_ses["home_tag_ses"] == ses].sort_values("hour")
    x = d["hour"] + (i - 1) * width
    ax.bar(x, d["share"] * 100, width=width, label=f"{SES_LABEL[ses]} (n={totals_ses[ses]:,})")
    ax.errorbar(x, d["share"] * 100,
                yerr=[(d["share"] - d["ci_low"]) * 100, (d["ci_high"] - d["share"]) * 100],
                fmt="none", ecolor="0.3", elinewidth=0.7, capsize=1.5)

ax.axvspan(EARLY_AFTERNOON_HOURS[0] - 0.5, EARLY_AFTERNOON_HOURS[1] + 0.5, color="grey", alpha=0.10, zorder=0)
ax.set_xlabel("Hour of arrival (local time)")
ax.set_ylabel("Share of the group's visits (%)")
ax.set_xticks(range(0, 24, 1))
ax.set_title("Weekday hourly profile of remote workers' third-place visits, by socio-economic "
             "status of the home area")
ax.legend(title="Visitor profile")
plt.show()

caption("""**Fig. B.** Weekday hourly profile of remote workers' visits to detected third places,
split by the socio-economic classification of the visitor's home statistical area. *Notes:* remote
workers only; shaded band marks 10:00-15:00; error bars are 95% Wilson binomial confidence
intervals.""")

### Hourly mean / share figures C2 and C–E, and late-afternoon window share

For each weekday visit metric below, the same layout is reused:

1. **Remote vs on-site workers** at detected third places only
2. **Detected third places vs other scored POIs** (all workers)
3. **Detected third places vs other scored POIs among remote workers only**

Each contrast is drawn twice: Tel Aviv-Yafo vs the rest of Gush Dan (2 panels), and civil vs commercial × location (4 panels). Figures use hours 07:00–20:00 only; hours with fewer than 30 visits omitted. No summary tables.

**C2** is the hourly **mean** distance from home (`dist_home_km`) with 95% normal CI bands. **C–E** are hourly **shares** of visit attributes (`near_home`, `is_long`, `from_home`) with Wilson 95% bands. **EA** is the overall share of visits arriving in late afternoon (14:00–18:59) — a window rate, so panels are group comparisons rather than hour curves.

Panels **5–6** repeat the third-vs-other contrast for **remote workers only**, by Tel Aviv-Yafo vs the rest of Gush Dan and by civil vs commercial × location.


In [ ]:
# Attach civil/commercial labels to the third-place weekday panel
df_weekday["poi_group"] = df_weekday["primarytype"].map(TYPE_GROUP)
n_unmap = int(df_weekday["poi_group"].isna().sum())
if n_unmap:
    print(
        f"Weekday third-place visits with unmapped primarytype (dropped in 4-panel splits): "
        f"{n_unmap:,} — "
        f"{df_weekday.loc[df_weekday['poi_group'].isna(), 'primarytype'].value_counts().head(8).to_dict()}"
    )

# One BigQuery aggregation for contrast (2): third places vs other scored POIs
_civil_sql = ", ".join(f"'{t}'" for t, g in TYPE_GROUP.items() if g == "Civil")
_comm_sql = ", ".join(f"'{t}'" for t, g in TYPE_GROUP.items() if g == "Commercial")

df_share_place = utils.read_query_to_dataframe(f"""
SELECT
  p.prediction_third_place = 1 AS is_third,
  up.work_schedule AS work_schedule,
  EXTRACT(HOUR FROM d.start_date_time) AS hour,
  STARTS_WITH(
    REGEXP_REPLACE(CAST(p.yishuv_stat_2022 AS STRING), r'\\.0$', ''),
    '{TEL_AVIV_SEMEI}'
  ) AS flag_tel_aviv,
  CASE
    WHEN LOWER(p.primarytype) IN ({_civil_sql}) THEN 'Civil'
    WHEN LOWER(p.primarytype) IN ({_comm_sql}) THEN 'Commercial'
    ELSE NULL
  END AS poi_group,
  COUNT(*) AS n,
  COUNTIF(d.dist_to_home < 1000) AS k_near_home,
  COUNTIF(d.duration_hours > {LONG_VISIT_HOURS}) AS k_is_long,
  COUNTIF(d.prev_flag_home AND d.time_diff_from_prev <= {FROM_PREV_MAX_GAP_H}) AS k_from_home,
  COUNTIF(EXTRACT(HOUR FROM d.start_date_time) BETWEEN {LATE_AFTERNOON_HOURS[0]} AND {LATE_AFTERNOON_HOURS[1]}) AS k_late_afternoon,
  AVG(SAFE_DIVIDE(d.dist_to_home, 1000)) AS mean_dist_home_km,
  STDDEV_SAMP(SAFE_DIVIDE(d.dist_to_home, 1000)) AS sd_dist_home_km,
  COUNTIF(d.dist_to_home IS NOT NULL) AS n_dist_home
FROM `{DWELLS_TABLE}` d
JOIN `{PU_TABLE}` p
  ON d.id = p.id
JOIN `{USER_PROFILES_TABLE}` up
  ON d.identifier = up.identifier
 AND d.month = up.month
WHERE NOT d.flag_home
  AND NOT d.flag_work
  AND d.day_type = 'weekday'
  AND d.month BETWEEN '{START_PERIOD}' AND '{END_PERIOD}'
  AND up.work_schedule IN {WORK_SCHEDULES}
  AND p.yishuv_stat_2022 IS NOT NULL
GROUP BY 1, 2, 3, 4, 5
""")

df_share_place["tel_aviv_label"] = np.where(
    df_share_place["flag_tel_aviv"], TLV_ORDER[0], TLV_ORDER[1]
)
print(
    "Third vs other scored POIs (weekday visit cells):\n",
    df_share_place.groupby("is_third")["n"].sum().rename(PLACE_TYPE_LABEL).to_string(),
)


def _place_flag_frame(flag_k_col, work_schedule=None):
    """Visit-cell frame for plot_hourly_share (pre-aggregated n / k).

    Pass work_schedule="remote_worker" to restrict to remote workers only.
    """
    out = df_share_place.dropna(subset=["tel_aviv_label"]).copy()
    if work_schedule is not None:
        out = out[out["work_schedule"] == work_schedule]
    out["k"] = out[flag_k_col]
    return out


def run_share_block(flag_col, ylabel, letter, metric_note):
    """Six hourly-share figures for one binary visit flag."""
    third = df_weekday.dropna(subset=["tel_aviv_label", flag_col]).copy()
    third = third[third["tel_aviv_label"].isin(TLV_ORDER)]

    print(
        f"Metric: {flag_col} | Categories: remote vs on-site (third places) / "
        f"third vs other POIs (all / remote-only) × Tel Aviv × civil/commercial | "
        f"X: share of weekday visits (%) with 95% Wilson CI"
    )

    plot_hourly_share(
        third,
        "work_schedule",
        list(WORK_SCHEDULES),
        SCHEDULE_LABEL,
        SCHEDULE_COLOR,
        ylabel,
        f"Fig. {letter}1 — {metric_note}, remote vs on-site, Tel Aviv vs rest of Gush Dan",
        f"{letter}1",
        flag_col=flag_col,
        hour_range=DAYTIME_HOURS,
        caption_text=(
            f"**Fig. {letter}1.** {metric_note} at detected third places, weekdays 2022-2023, "
            f"remote versus on-site workers. Panels: Tel Aviv-Yafo versus the rest of Gush Dan "
            f"(`semel_yishuv` = 5000). *Notes:* unit of analysis is visits; hours with n < 30 "
            f"omitted; shaded bands are 95% Wilson binomial confidence intervals."
        ),
    )
    plot_hourly_share(
        third.dropna(subset=["poi_group"]),
        "work_schedule",
        list(WORK_SCHEDULES),
        SCHEDULE_LABEL,
        SCHEDULE_COLOR,
        ylabel,
        f"Fig. {letter}2 — {metric_note}, remote vs on-site, civil/commercial × location",
        f"{letter}2",
        flag_col=flag_col,
        facet_row_col="poi_group",
        hour_range=DAYTIME_HOURS,
        caption_text=(
            f"**Fig. {letter}2.** Same metric as Fig. {letter}1, split by civil versus commercial "
            f"POI type and Tel Aviv-Yafo versus the rest of Gush Dan. Civil = library, museum, "
            f"community centre, cultural centre; commercial = cafe, bakery, bookshop and related "
            f"types. *Notes:* unmapped primary types omitted; hours restricted to {DAYTIME_HOURS[0]:02d}:00–{DAYTIME_HOURS[1]:02d}:00; hours with n < 30 omitted; "
            f"95% Wilson bands."
        ),
    )

    place = _place_flag_frame({
        "near_home": "k_near_home",
        "is_long": "k_is_long",
        "from_home": "k_from_home",
        "is_late_afternoon": "k_late_afternoon",
    }[flag_col])

    plot_hourly_share(
        place,
        "is_third",
        PLACE_TYPE_ORDER,
        PLACE_TYPE_LABEL,
        PLACE_TYPE_COLOR,
        ylabel,
        f"Fig. {letter}3 — {metric_note}, third places vs other scored POIs, by location",
        f"{letter}3",
        hour_range=DAYTIME_HOURS,
        caption_text=(
            f"**Fig. {letter}3.** {metric_note}, detected third places versus other PU-scored POIs, "
            f"Tel Aviv-Yafo versus the rest of Gush Dan. Same weekday visitor sample "
            f"(remote and on-site workers). *Notes:* home and work dwells excluded; hours with "
            f"n < 30 omitted; 95% Wilson bands."
        ),
    )
    plot_hourly_share(
        place.dropna(subset=["poi_group"]),
        "is_third",
        PLACE_TYPE_ORDER,
        PLACE_TYPE_LABEL,
        PLACE_TYPE_COLOR,
        ylabel,
        f"Fig. {letter}4 — {metric_note}, third vs other POIs, civil/commercial × location",
        f"{letter}4",
        facet_row_col="poi_group",
        hour_range=DAYTIME_HOURS,
        caption_text=(
            f"**Fig. {letter}4.** Same metric as Fig. {letter}3, split by civil versus commercial "
            f"POI type and Tel Aviv-Yafo versus the rest of Gush Dan. *Notes:* unmapped primary "
            f"types omitted; hours restricted to {DAYTIME_HOURS[0]:02d}:00–{DAYTIME_HOURS[1]:02d}:00; hours with n < 30 omitted; 95% Wilson bands."
        ),
    )



    # Remote workers only: third places vs other scored POIs
    place_rw = _place_flag_frame({
        "near_home": "k_near_home",
        "is_long": "k_is_long",
        "from_home": "k_from_home",
        "is_late_afternoon": "k_late_afternoon",
    }[flag_col], work_schedule="remote_worker")

    plot_hourly_share(
        place_rw,
        "is_third",
        PLACE_TYPE_ORDER,
        PLACE_TYPE_LABEL,
        PLACE_TYPE_COLOR,
        ylabel,
        f"Fig. {letter}5 — {metric_note}, third vs other POIs (remote workers only), by location",
        f"{letter}5",
        hour_range=DAYTIME_HOURS,
        caption_text=(
            f"**Fig. {letter}5.** {metric_note}, detected third places versus other PU-scored POIs "
            f"among remote workers only. Panels: Tel Aviv-Yafo versus the rest of Gush Dan. "
            f"*Notes:* home and work dwells excluded; hours with n < 30 omitted; 95% Wilson bands."
        ),
    )
    plot_hourly_share(
        place_rw.dropna(subset=["poi_group"]),
        "is_third",
        PLACE_TYPE_ORDER,
        PLACE_TYPE_LABEL,
        PLACE_TYPE_COLOR,
        ylabel,
        f"Fig. {letter}6 — {metric_note}, third vs other POIs (remote workers only), "
        f"civil/commercial × location",
        f"{letter}6",
        facet_row_col="poi_group",
        hour_range=DAYTIME_HOURS,
        caption_text=(
            f"**Fig. {letter}6.** Same metric as Fig. {letter}5, split by civil versus commercial "
            f"POI type and Tel Aviv-Yafo versus the rest of Gush Dan. Remote workers only. "
            f"*Notes:* unmapped primary types omitted; hours restricted to "
            f"{DAYTIME_HOURS[0]:02d}:00–{DAYTIME_HOURS[1]:02d}:00; hours with n < 30 omitted; "
            f"95% Wilson bands."
        ),
    )



def plot_hourly_mean(
    df,
    series_col,
    series_order,
    series_label,
    series_color,
    ylabel,
    title,
    fig_id,
    value_col=None,
    facet_row_col=None,
    facet_col_col="tel_aviv_label",
    facet_row_order=None,
    facet_col_order=None,
    min_hour=MIN_HOUR_N,
    hour_range=None,
    caption_text=None,
):
    """Hourly mean of a continuous visit metric, with 95% normal CI bands.

    Pass visit-level rows + value_col, or a pre-aggregated frame with columns
    mean, sd, n (optionally ci_low / ci_high) already at the panel grain.
    hour_range=(lo, hi) keeps only hours lo..hi inclusive (e.g. daytime 7–20).
    """
    facet_col_order = list(facet_col_order or TLV_ORDER)
    if facet_row_col is None:
        facet_row_order = [None]
        nrows, ncols = 1, len(facet_col_order)
    else:
        facet_row_order = list(facet_row_order or POI_GROUP_ORDER)
        nrows, ncols = len(facet_row_order), len(facet_col_order)

    need = [series_col, "hour", facet_col_col]
    if facet_row_col is not None:
        need.append(facet_row_col)
    panel = df.dropna(subset=need).copy()
    panel = panel[panel[facet_col_col].isin(facet_col_order)]
    if facet_row_col is not None:
        panel = panel[panel[facet_row_col].isin(facet_row_order)]
    panel = panel[panel[series_col].isin(series_order)]
    if hour_range is not None:
        lo, hi = hour_range
        panel = panel[panel["hour"].between(lo, hi)]

    group_cols = [c for c in [facet_row_col, facet_col_col, series_col, "hour"] if c is not None]
    if value_col is not None:
        agg = mean_by(panel, group_cols, value_col)
    else:
        if not {"mean", "sd", "n"}.issubset(panel.columns):
            raise ValueError("Pre-aggregated input needs columns mean, sd, n (or pass value_col)")
        # Already at (facet × series × hour); do not re-average means across cells.
        keep = group_cols + ["mean", "sd", "n"]
        for optional in ("ci_low", "ci_high"):
            if optional in panel.columns:
                keep.append(optional)
        agg = panel[keep].drop_duplicates(subset=group_cols).copy()
        if not {"ci_low", "ci_high"}.issubset(agg.columns):
            se = agg["sd"] / np.sqrt(agg["n"].astype(float))
            agg = agg.assign(ci_low=agg["mean"] - 1.96 * se, ci_high=agg["mean"] + 1.96 * se)
    agg = agg[agg["n"] >= min_hour]

    fig, axes = plt.subplots(
        nrows, ncols, figsize=(7.0 * ncols, 4.2 * nrows),
        sharex=True, sharey=True, squeeze=False,
    )
    for i, row_key in enumerate(facet_row_order):
        for j, col_key in enumerate(facet_col_order):
            ax = axes[i][j]
            sub = agg[agg[facet_col_col] == col_key]
            if facet_row_col is not None:
                sub = sub[sub[facet_row_col] == row_key]
            n_panel = int(sub.groupby(series_col)["n"].sum().sum()) if len(sub) else 0
            for series in series_order:
                d = sub[sub[series_col] == series].sort_values("hour")
                if d.empty:
                    continue
                color = series_color[series]
                label = series_label[series]
                ax.plot(d["hour"], d["mean"], marker="o", ms=4, color=color, label=label)
                ax.fill_between(
                    d["hour"], d["ci_low"], d["ci_high"],
                    color=color, alpha=0.20,
                )
            ax.axvspan(EARLY_AFTERNOON_HOURS[0] - 0.5, EARLY_AFTERNOON_HOURS[1] + 0.5, color="grey", alpha=0.10, zorder=0)
            if facet_row_col is None:
                ax.set_title(f"{col_key} (n={n_panel:,})")
            else:
                ax.set_title(f"{row_key} · {col_key} (n={n_panel:,})")
            ax.set_xticks(
                range(hour_range[0], hour_range[1] + 1, 2) if hour_range else range(0, 24, 2)
            )
            if i == nrows - 1:
                ax.set_xlabel("Hour of arrival (local time)")
            if j == 0:
                ax.set_ylabel(ylabel)

    axes[0][0].legend(loc="upper left", fontsize=8)
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()
    if caption_text:
        caption(caption_text)
    return agg

def _place_mean_frame(work_schedule=None):
    """Visit-cell frame for plot_hourly_mean (pre-aggregated mean / sd / n).

    Pass work_schedule="remote_worker" to restrict to remote workers only.
    """
    out = df_share_place.dropna(subset=["tel_aviv_label"]).copy()
    if work_schedule is not None:
        out = out[out["work_schedule"] == work_schedule]
    out["mean"] = out["mean_dist_home_km"]
    out["sd"] = out["sd_dist_home_km"]
    out["n"] = out["n_dist_home"]
    return out


def run_mean_block(value_col, ylabel, letter, metric_note):
    """Six hourly-mean figures for one continuous visit metric (same contrasts as C–E)."""
    third = df_weekday.dropna(subset=["tel_aviv_label", value_col]).copy()
    third = third[third["tel_aviv_label"].isin(TLV_ORDER)]

    print(
        f"Metric: {value_col} | Categories: remote vs on-site (third places) / "
        f"third vs other POIs (all / remote-only) × Tel Aviv × civil/commercial | "
        f"X: mean with 95% normal CI"
    )

    plot_hourly_mean(
        third,
        "work_schedule",
        list(WORK_SCHEDULES),
        SCHEDULE_LABEL,
        SCHEDULE_COLOR,
        ylabel,
        f"Fig. {letter}1 — {metric_note}, remote vs on-site, Tel Aviv vs rest of Gush Dan",
        f"{letter}1",
        value_col=value_col,
        hour_range=DAYTIME_HOURS,
        caption_text=(
            f"**Fig. {letter}1.** {metric_note} at detected third places, weekdays 2022-2023, "
            f"remote versus on-site workers. Panels: Tel Aviv-Yafo versus the rest of Gush Dan "
            f"(`semel_yishuv` = 5000). *Notes:* unit of analysis is visits; hours restricted to "
            f"{DAYTIME_HOURS[0]:02d}:00–{DAYTIME_HOURS[1]:02d}:00; hours with n < 30 omitted; "
            f"shaded bands are 95% normal confidence intervals."
        ),
    )
    plot_hourly_mean(
        third.dropna(subset=["poi_group"]),
        "work_schedule",
        list(WORK_SCHEDULES),
        SCHEDULE_LABEL,
        SCHEDULE_COLOR,
        ylabel,
        f"Fig. {letter}2 — {metric_note}, remote vs on-site, civil/commercial × location",
        f"{letter}2",
        value_col=value_col,
        facet_row_col="poi_group",
        hour_range=DAYTIME_HOURS,
        caption_text=(
            f"**Fig. {letter}2.** Same metric as Fig. {letter}1, split by civil versus commercial "
            f"POI type and Tel Aviv-Yafo versus the rest of Gush Dan. *Notes:* unmapped primary "
            f"types omitted; hours restricted to {DAYTIME_HOURS[0]:02d}:00–{DAYTIME_HOURS[1]:02d}:00; "
            f"hours with n < 30 omitted; 95% normal CI bands."
        ),
    )

    place = _place_mean_frame()

    plot_hourly_mean(
        place,
        "is_third",
        PLACE_TYPE_ORDER,
        PLACE_TYPE_LABEL,
        PLACE_TYPE_COLOR,
        ylabel,
        f"Fig. {letter}3 — {metric_note}, third places vs other scored POIs, by location",
        f"{letter}3",
        hour_range=DAYTIME_HOURS,
        caption_text=(
            f"**Fig. {letter}3.** {metric_note}, detected third places versus other PU-scored POIs, "
            f"Tel Aviv-Yafo versus the rest of Gush Dan. Same weekday visitor sample "
            f"(remote and on-site workers). *Notes:* home and work dwells excluded; hours with "
            f"n < 30 omitted; 95% normal CI bands."
        ),
    )
    plot_hourly_mean(
        place.dropna(subset=["poi_group"]),
        "is_third",
        PLACE_TYPE_ORDER,
        PLACE_TYPE_LABEL,
        PLACE_TYPE_COLOR,
        ylabel,
        f"Fig. {letter}4 — {metric_note}, third vs other POIs, civil/commercial × location",
        f"{letter}4",
        facet_row_col="poi_group",
        hour_range=DAYTIME_HOURS,
        caption_text=(
            f"**Fig. {letter}4.** Same metric as Fig. {letter}3, split by civil versus commercial "
            f"POI type and Tel Aviv-Yafo versus the rest of Gush Dan. *Notes:* unmapped primary "
            f"types omitted; hours restricted to {DAYTIME_HOURS[0]:02d}:00–{DAYTIME_HOURS[1]:02d}:00; "
            f"hours with n < 30 omitted; 95% normal CI bands."
        ),
    )


    # Remote workers only: third places vs other scored POIs
    place_rw = _place_mean_frame(work_schedule="remote_worker")

    plot_hourly_mean(
        place_rw,
        "is_third",
        PLACE_TYPE_ORDER,
        PLACE_TYPE_LABEL,
        PLACE_TYPE_COLOR,
        ylabel,
        f"Fig. {letter}5 — {metric_note}, third vs other POIs (remote workers only), by location",
        f"{letter}5",
        hour_range=DAYTIME_HOURS,
        caption_text=(
            f"**Fig. {letter}5.** {metric_note}, detected third places versus other PU-scored POIs "
            f"among remote workers only. Panels: Tel Aviv-Yafo versus the rest of Gush Dan. "
            f"*Notes:* home and work dwells excluded; hours with n < 30 omitted; 95% normal CI bands."
        ),
    )
    plot_hourly_mean(
        place_rw.dropna(subset=["poi_group"]),
        "is_third",
        PLACE_TYPE_ORDER,
        PLACE_TYPE_LABEL,
        PLACE_TYPE_COLOR,
        ylabel,
        f"Fig. {letter}6 — {metric_note}, third vs other POIs (remote workers only), "
        f"civil/commercial × location",
        f"{letter}6",
        facet_row_col="poi_group",
        hour_range=DAYTIME_HOURS,
        caption_text=(
            f"**Fig. {letter}6.** Same metric as Fig. {letter}5, split by civil versus commercial "
            f"POI type and Tel Aviv-Yafo versus the rest of Gush Dan. Remote workers only. "
            f"*Notes:* unmapped primary types omitted; hours restricted to "
            f"{DAYTIME_HOURS[0]:02d}:00–{DAYTIME_HOURS[1]:02d}:00; hours with n < 30 omitted; "
            f"95% normal CI bands."
        ),
    )




print("Hourly-share helpers ready.")


def plot_overall_share(
    df,
    series_col,
    series_order,
    series_label,
    series_color,
    ylabel,
    title,
    fig_id,
    flag_col=None,
    facet_row_col=None,
    facet_col_col="tel_aviv_label",
    facet_row_order=None,
    facet_col_order=None,
    min_cell=200,
    caption_text=None,
):
    """Overall share of a binary visit flag by group, with Wilson CIs.

    Use for window rates (e.g. late afternoon) that are not meaningful as
    P(flag | hour). Visit-level + flag_col, or pre-aggregated n / k.
    """
    facet_col_order = list(facet_col_order or TLV_ORDER)
    if facet_row_col is None:
        facet_row_order = [None]
        nrows, ncols = 1, len(facet_col_order)
    else:
        facet_row_order = list(facet_row_order or POI_GROUP_ORDER)
        nrows, ncols = len(facet_row_order), len(facet_col_order)

    need = [series_col, facet_col_col]
    if facet_row_col is not None:
        need.append(facet_row_col)
    panel = df.dropna(subset=need).copy()
    panel = panel[panel[facet_col_col].isin(facet_col_order)]
    if facet_row_col is not None:
        panel = panel[panel[facet_row_col].isin(facet_row_order)]
    panel = panel[panel[series_col].isin(series_order)]

    group_cols = [c for c in [facet_row_col, facet_col_col, series_col] if c is not None]
    if flag_col is not None:
        agg = share_by(panel, group_cols, flag_col)
    else:
        if not {"n", "k"}.issubset(panel.columns):
            raise ValueError("Pre-aggregated input needs columns n and k (or pass flag_col)")
        agg = (
            panel.groupby(group_cols, observed=True)[["n", "k"]]
            .sum()
            .reset_index()
        )
        agg = add_wilson_ci(agg)
    agg = agg[agg["n"] >= min_cell]
    if agg.empty:
        print(f"No cells with n >= {min_cell} for {title}")
        return agg

    y = np.arange(len(series_order))
    pad = max(1.5, (agg["ci_high"].max() - agg["ci_low"].min()) * 100 * 0.18)
    xmin = float(agg["ci_low"].min()) * 100 - pad
    xmax = float(agg["ci_high"].max()) * 100 + pad

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(5.8 * ncols, max(2.6, 0.55 * len(series_order) + 2.1) * nrows),
        sharex=True, sharey=True, squeeze=False,
    )
    for i, row_key in enumerate(facet_row_order):
        for j, col_key in enumerate(facet_col_order):
            ax = axes[i][j]
            sub = agg[agg[facet_col_col] == col_key]
            if facet_row_col is not None:
                sub = sub[sub[facet_row_col] == row_key]
            n_panel = int(sub["n"].sum()) if len(sub) else 0
            panel_share = (sub["k"].sum() / sub["n"].sum() * 100) if n_panel else np.nan
            for yi, series in enumerate(series_order):
                row = sub[sub[series_col] == series]
                if row.empty:
                    continue
                r = row.iloc[0]
                ax.errorbar(
                    r["share"] * 100, yi,
                    xerr=[[(r["share"] - r["ci_low"]) * 100],
                          [(r["ci_high"] - r["share"]) * 100]],
                    fmt="o", ms=6, color=series_color[series],
                    ecolor=series_color[series], elinewidth=1.4, capsize=2.5,
                    label=series_label[series],
                )
            if np.isfinite(panel_share):
                ax.axvline(
                    panel_share, ls="--", lw=1.2, color="#718096",
                    label=f"Panel mean ({panel_share:.1f}%)",
                )
            ax.set_yticks(y)
            ax.set_yticklabels([series_label[s] for s in series_order])
            ax.set_ylim(len(series_order) - 0.5, -0.5)
            ax.set_xlim(xmin, xmax)
            if facet_row_col is None:
                ax.set_title(f"{col_key} (n={n_panel:,})")
            else:
                ax.set_title(f"{row_key} · {col_key} (n={n_panel:,})")
            if i == nrows - 1:
                ax.set_xlabel(ylabel)

    axes[0][-1].legend(fontsize=8, loc="best")
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()
    if caption_text:
        caption(caption_text)
    return agg


def run_overall_share_block(flag_col, ylabel, letter, metric_note):
    """Six overall-share figures for one binary visit flag (same contrasts as C–E)."""
    third = df_weekday.dropna(subset=["tel_aviv_label", flag_col]).copy()
    if flag_col == "near_home":
        third = third.dropna(subset=["dist_home_km"])
    third = third[third["tel_aviv_label"].isin(TLV_ORDER)]

    print(
        f"Metric: {flag_col} | Categories: remote vs on-site (third places) / "
        f"third vs other POIs (all / remote-only) × Tel Aviv × civil/commercial | "
        f"X: share of weekday visits (%) with 95% Wilson CI"
    )

    plot_overall_share(
        third,
        "work_schedule",
        list(WORK_SCHEDULES),
        SCHEDULE_LABEL,
        SCHEDULE_COLOR,
        ylabel,
        f"Fig. {letter}1 — {metric_note}, remote vs on-site, Tel Aviv vs rest of Gush Dan",
        f"{letter}1",
        flag_col=flag_col,
        caption_text=(
            f"**Fig. {letter}1.** {metric_note} at detected third places, weekdays 2022-2023, "
            f"remote versus on-site workers. Panels: Tel Aviv-Yafo versus the rest of Gush Dan "
            f"(`semel_yishuv` = 5000). *Notes:* unit of analysis is visits; cells with n < 200 "
            f"omitted; points and error bars are 95% Wilson binomial confidence intervals; "
            f"x-axis is zoomed to the range of the estimates (not 0–100)."
            f"{'' if flag_col != 'near_home' else (
                f' Visits with missing distance to home are dropped in Figs {letter}1–{letter}2; '
                f'in Figs {letter}3–{letter}4 they remain in the denominator (not within 1 km).'
            )}"
        ),
    )
    plot_overall_share(
        third.dropna(subset=["poi_group"]),
        "work_schedule",
        list(WORK_SCHEDULES),
        SCHEDULE_LABEL,
        SCHEDULE_COLOR,
        ylabel,
        f"Fig. {letter}2 — {metric_note}, remote vs on-site, civil/commercial × location",
        f"{letter}2",
        flag_col=flag_col,
        facet_row_col="poi_group",
        caption_text=(
            f"**Fig. {letter}2.** Same metric as Fig. {letter}1, split by civil versus commercial "
            f"POI type and Tel Aviv-Yafo versus the rest of Gush Dan. *Notes:* unmapped primary "
            f"types omitted; cells with n < 200 omitted; 95% Wilson intervals."
        ),
    )

    place = _place_flag_frame({
        "near_home": "k_near_home",
        "is_long": "k_is_long",
        "from_home": "k_from_home",
        "is_late_afternoon": "k_late_afternoon",
    }[flag_col])

    plot_overall_share(
        place,
        "is_third",
        PLACE_TYPE_ORDER,
        PLACE_TYPE_LABEL,
        PLACE_TYPE_COLOR,
        ylabel,
        f"Fig. {letter}3 — {metric_note}, third places vs other scored POIs, by location",
        f"{letter}3",
        caption_text=(
            f"**Fig. {letter}3.** {metric_note}, detected third places versus other PU-scored POIs, "
            f"Tel Aviv-Yafo versus the rest of Gush Dan. *Notes:* home and work dwells excluded; "
            f"cells with n < 200 omitted; 95% Wilson intervals; x-axis zoomed to the estimates."
        ),
    )
    plot_overall_share(
        place.dropna(subset=["poi_group"]),
        "is_third",
        PLACE_TYPE_ORDER,
        PLACE_TYPE_LABEL,
        PLACE_TYPE_COLOR,
        ylabel,
        f"Fig. {letter}4 — {metric_note}, third vs other POIs, civil/commercial × location",
        f"{letter}4",
        facet_row_col="poi_group",
        caption_text=(
            f"**Fig. {letter}4.** Same metric as Fig. {letter}3, split by civil versus commercial "
            f"POI type and Tel Aviv-Yafo versus the rest of Gush Dan. *Notes:* unmapped primary "
            f"types omitted; cells with n < 200 omitted; 95% Wilson intervals."
        ),
    )

    place_rw = _place_flag_frame({
        "near_home": "k_near_home",
        "is_long": "k_is_long",
        "from_home": "k_from_home",
        "is_late_afternoon": "k_late_afternoon",
    }[flag_col], work_schedule="remote_worker")

    plot_overall_share(
        place_rw,
        "is_third",
        PLACE_TYPE_ORDER,
        PLACE_TYPE_LABEL,
        PLACE_TYPE_COLOR,
        ylabel,
        f"Fig. {letter}5 — {metric_note}, third vs other POIs (remote workers only), by location",
        f"{letter}5",
        caption_text=(
            f"**Fig. {letter}5.** {metric_note}, detected third places versus other PU-scored POIs "
            f"among remote workers only, Tel Aviv-Yafo versus the rest of Gush Dan. *Notes:* "
            f"home and work dwells excluded; cells with n < 200 omitted; 95% Wilson intervals; "
            f"x-axis zoomed to the estimates."
        ),
    )
    plot_overall_share(
        place_rw.dropna(subset=["poi_group"]),
        "is_third",
        PLACE_TYPE_ORDER,
        PLACE_TYPE_LABEL,
        PLACE_TYPE_COLOR,
        ylabel,
        f"Fig. {letter}6 — {metric_note}, third vs other POIs (remote workers only), "
        f"civil/commercial × location",
        f"{letter}6",
        facet_row_col="poi_group",
        caption_text=(
            f"**Fig. {letter}6.** Same metric as Fig. {letter}5, split by civil versus commercial "
            f"POI type and Tel Aviv-Yafo versus the rest of Gush Dan. Remote workers only. "
            f"*Notes:* unmapped primary types omitted; cells with n < 200 omitted; "
            f"95% Wilson intervals."
        ),
    )



### Figure C2 — Mean distance from home

```text
Metric: dist_home_km (straight-line home-to-place, km)
Categories: (1) remote vs on-site at third places; (2) third vs other (all workers); (3) third vs other (remote workers only)
Facets: Tel Aviv-Yafo vs rest of Gush Dan; civil/commercial × location
X: mean km with 95% normal CI
```


In [ ]:
run_mean_block(
    "dist_home_km",
    "Mean distance from home (km)",
    "C2",
    "Mean weekday distance from home",
)


### Figure C — Visits within 1 km of home

```text
Metric: near_home (dist_home_km < 1)
Categories: (1) remote vs on-site at third places; (2) third vs other (all workers); (3) third vs other (remote workers only)
Facets: Tel Aviv-Yafo vs rest of Gush Dan; civil/commercial × location
X: share of weekday visits (%) with 95% Wilson CI
```


In [ ]:
run_share_block(
    "near_home",
    "Visits within 1 km of home (%)",
    "C",
    "Share of weekday visits within 1 km of home",
)


### Figure D — Long visits

```text
Metric: is_long (duration > 1 hour)
Categories: (1) remote vs on-site at third places; (2) third vs other (all workers); (3) third vs other (remote workers only)
Facets: Tel Aviv-Yafo vs rest of Gush Dan; civil/commercial × location
X: share of weekday visits (%) with 95% Wilson CI
```


In [ ]:
run_share_block(
    "is_long",
    f"Visits lasting more than {LONG_VISIT_HOURS:g} hour (%)",
    "D",
    "Share of weekday visits lasting more than one hour",
)


### Figure E — Arriving from home

```text
Metric: from_home (previous dwell was home, gap ≤ 2 h)
Categories: (1) remote vs on-site at third places; (2) third vs other (all workers); (3) third vs other (remote workers only)
Facets: Tel Aviv-Yafo vs rest of Gush Dan; civil/commercial × location
X: share of weekday visits (%) with 95% Wilson CI
```


In [ ]:
run_share_block(
    "from_home",
    "Visits arriving directly from home (%)",
    "E",
    "Share of weekday visits arriving directly from home",
)


### Figure EA — Late-afternoon visits (14:00–18:59)

Share of weekday visits arriving in late afternoon (14:00–18:59), the window between early afternoon and evening.

```text
Metric: is_late_afternoon (hour in 14–18)
Categories: (1) remote vs on-site at third places; (2) third vs other (all workers); (3) third vs other (remote workers only)
Facets: Tel Aviv-Yafo vs rest of Gush Dan; civil/commercial × location
X: share of weekday visits (%) with 95% Wilson CI
```


In [ ]:
run_overall_share_block(
    "is_late_afternoon",
    f"Visits arriving {LATE_AFTERNOON_HOURS[0]}:00–{LATE_AFTERNOON_HOURS[1]}:59 (%)",
    "EA",
    f"Share of weekday visits arriving {LATE_AFTERNOON_HOURS[0]}:00–{LATE_AFTERNOON_HOURS[1]}:59",
)


### Figures F and G — Tel Aviv vs rest of Gush Dan, and same-category density


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

ax = axes[0]
counts = df_places["tel_aviv_label"].value_counts().reindex(TLV_ORDER)
ax.bar(counts.index.astype(str), counts.values, color="#4a5568")
for i, v in enumerate(counts.fillna(0).values):
    ax.text(i, v, f"{int(v)}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Detected third places (count)")
ax.set_xlabel("Location of the place")
ax.set_title("Where detected third places are located")

ax = axes[1]
rw = share_by(df_weekday.assign(is_rw=df_weekday["work_schedule"] == "remote_worker")
              .dropna(subset=["tel_aviv_label"]),
              ["tel_aviv_label"], "is_rw")
rw = rw.set_index("tel_aviv_label").reindex(TLV_ORDER).dropna(subset=["share"])
ax.bar(rw.index.astype(str), rw["share"] * 100, color="#2b6cb0",
       yerr=[(rw["share"] - rw["ci_low"]) * 100, (rw["ci_high"] - rw["share"]) * 100],
       capsize=3, error_kw={"elinewidth": 0.9, "ecolor": "0.3"})
ax.axhline(RW_BASELINE * 100, ls="--", lw=1, color="0.4",
           label=f"Panel average ({RW_BASELINE:.0%})")
ax.set_ylabel("Remote workers' share of\nremote + on-site visits (%)")
ax.set_xlabel("Location of the place")
ax.set_title("Visitor mix, Tel Aviv-Yafo vs rest of Gush Dan")
ax.set_ylim(0, 70)
ax.legend(fontsize=8, loc="lower right")

plt.show()

caption("""**Fig. F.** Detected third places in Tel Aviv-Yafo versus the rest of Gush Dan, 2022-2023
(`semel_yishuv` = 5000). Left: number of detected places. Right: remote workers as a share of
visits by users with a determined work schedule, with 95% Wilson binomial confidence intervals;
the dashed line is the panel average. *Notes:* the denominator is remote plus on-site workers
only, so the level is not the remote-worker share of all visitors (about 19% at the average
detected place) and only the Tel Aviv versus rest gap should be read. Places without a
statistical-area code are excluded from both panels.""")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))

ax = axes[0]
ax.hist(df_places["n_same_primarytype_1km"].dropna(), bins=30, color="#4a5568")
med = df_places["n_same_primarytype_1km"].median()
ax.axvline(med, ls="--", color="#dd6b20", label=f"Median = {med:.0f}")
ax.set_xlabel(f"POIs of the same category within {DENSITY_RADIUS_M/1000:.0f} km (count)")
ax.set_ylabel("Detected third places (count)")
ax.set_title("Local supply of the same amenity category")
ax.legend()

ax = axes[1]
dens = df_weekday.dropna(subset=["density_tertile"]).copy()
rw_dens = share_by(dens.assign(is_rw=dens["work_schedule"] == "remote_worker"),
                   ["density_tertile"], "is_rw")
long_dens = share_by(dens, ["density_tertile"], "is_long")

x = np.arange(len(rw_dens))
width = 0.38
ax.bar(x - width / 2, rw_dens["share"] * 100, width, color="#2b6cb0",
       label="Remote workers' share of remote + on-site visits",
       yerr=[(rw_dens["share"] - rw_dens["ci_low"]) * 100,
             (rw_dens["ci_high"] - rw_dens["share"]) * 100],
       capsize=3, error_kw={"elinewidth": 0.9, "ecolor": "0.3"})
ax.bar(x + width / 2, long_dens["share"] * 100, width, color="#dd6b20",
       label="Visits longer than 1 hour",
       yerr=[(long_dens["share"] - long_dens["ci_low"]) * 100,
             (long_dens["ci_high"] - long_dens["share"]) * 100],
       capsize=3, error_kw={"elinewidth": 0.9, "ecolor": "0.3"})
ax.set_xticks(x)
ax.set_xticklabels(rw_dens["density_tertile"].astype(str))
ax.set_ylabel("Share of weekday visits (%)")
ax.set_xlabel(f"Tertile of same-category POIs within {DENSITY_RADIUS_M/1000:.0f} km")
ax.set_title("Visitor mix and lingering by local amenity density")
ax.set_ylim(0, 70)
ax.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2)

plt.show()

caption("""**Fig. G.** Same-category amenity density around detected third places, 2022-2023. Left:
distribution of the number of Google POIs sharing the place's primary category within 1 km. Right:
remote workers as a share of visits by users with a determined work schedule, and the share of
visits lasting over an hour, by tertile of that density, with 95% Wilson binomial confidence
intervals. *Notes:* density is measured against the full Google POI universe for the metropolitan
area, not only detected third places; as in Fig. F the remote-worker share is computed over remote
plus on-site workers, so only differences across tertiles should be read.""")

## 6. Visitor profiles

`user_profiles_monthly` carries three families of attributes per user-month, and each asks a
different question of the third-place literature:

* **Work context** — `work_context_type`, `work_neighborhood_type`, `employment_zone`. Where the
  user's work location actually is. A user whose work location is classified `residential` is
  working from home, which is the sharpest available proxy for remote work independent of the
  `work_schedule` label.
* **Home-area census tags** — `home_tag_ses`, `home_tag_occupation_proxy`, `home_tag_religion` (Census tags), plus
  CBS Census 2022 home-SA **IT share** (`IndstJ_pcnt`) and **academic occupation**
  (`Occup2_pcnt`) as in notebook 3302 — split at the median (below / above) for composition charts.
  Education and car-availability tags are not used here.
* **Behavioural flags** — `FLAG_PARENT`, `FLAG_STUDENT`, `FLAG_MEDICAL`, `FLAG_RELIGIOUS`,
  `FLAG_LIKE_*`. What else the user does in a month. These capture the competing obligations that
  the literature treats as the main constraint on third-place participation.

The audit below checks each attribute before use, because two of them are unusable as stored.


In [ ]:
CATEGORICAL_ATTRS = ["work_context_type", "work_neighborhood_type", "home_neighborhood_type",
                     "home_tag_ses", "home_tag_education", "home_tag_occupation_proxy",
                     "home_tag_mobility", "home_tag_religion", "home_tag_lifecycle"]
BINARY_ATTRS = ["employment_zone", "FLAG_PARENT", "FLAG_STUDENT", "FLAG_MEDICAL",
                "FLAG_RELIGIOUS", "like_cafes", "like_shopping", "like_outdoor",
                "flag_reliable_min_5days"]

rows = []
for col in CATEGORICAL_ATTRS:
    vc = df_visits[col].value_counts(normalize=True, dropna=True)
    rows.append({"attribute": col, "type": "categorical", "levels": len(vc),
                 "modal_level": vc.index[0], "modal_share": vc.iloc[0],
                 "missing": df_visits[col].isna().mean()})
for col in BINARY_ATTRS:
    rows.append({"attribute": col, "type": "binary", "levels": 2,
                 "modal_level": "True", "modal_share": df_visits[col].mean(),
                 "missing": 0.0})

audit = pd.DataFrame(rows)
# an attribute is only informative if the rarer side is big enough to estimate a share on
audit["minority_share"] = np.where(audit["type"] == "binary",
                                   np.minimum(audit["modal_share"], 1 - audit["modal_share"]),
                                   1 - audit["modal_share"])
audit["usable"] = np.where(audit["minority_share"] < 0.005, "no - degenerate",
                           np.where(audit["minority_share"] < 0.02, "weak - small cell", "yes"))
audit = audit.sort_values(["usable", "minority_share"]).round(4)

In [ ]:
# Home-SA Census 2022 occupation / industry shares (same join as 3302).
# Median split (below / above) is used in Fig. H / H2 as categorical profile groups.
# If df_visits was loaded before home_yishuv_stat_2022 was added to the visits
# query, this cell pulls that column from user profiles and merges it in.

df_census_occ = utils.read_query_to_dataframe(f"""
SELECT
  CAST(sa.yishuv_stat_2022 AS STRING) AS home_yishuv_stat_2022,
  c.IndstJ_pcnt AS it_pct,
  c.Occup2_pcnt AS academic_occup_pct
FROM `{CENSUS_TABLE}` sa
LEFT JOIN `{CENSUS_2022_TABLE}` c
  ON c.LocalityCode = sa.semel_yishuv
 AND c.StatArea = sa.stat_2022
""")

for col in ("it_pct", "academic_occup_pct"):
    df_census_occ[col] = pd.to_numeric(df_census_occ[col], errors="coerce")

print(
    "Census SA coverage:",
    f"{df_census_occ['home_yishuv_stat_2022'].nunique():,} SAs | "
    f"missing it_pct={df_census_occ['it_pct'].isna().mean():.1%} | "
    f"missing academic_occup_pct={df_census_occ['academic_occup_pct'].isna().mean():.1%}"
)
print(df_census_occ[["it_pct", "academic_occup_pct"]].describe().round(2))


def _ensure_home_yishuv(df):
    """Attach home_yishuv_stat_2022 from profiles if the visits frame lacks it."""
    out = df.copy()
    if "home_yishuv_stat_2022" in out.columns and out["home_yishuv_stat_2022"].notna().any():
        out["home_yishuv_stat_2022"] = (
            out["home_yishuv_stat_2022"].astype("string").str.replace(r"\.0$", "", regex=True)
        )
        return out

    print(
        "home_yishuv_stat_2022 missing on visits frame — loading from "
        f"`{USER_PROFILES_TABLE}` (re-run the visits query later to bake it in)."
    )
    df_home = utils.read_query_to_dataframe(f"""
    SELECT
      identifier,
      month,
      CAST(home_yishuv_stat_2022 AS STRING) AS home_yishuv_stat_2022
    FROM `{USER_PROFILES_TABLE}`
    WHERE month BETWEEN '{START_PERIOD}' AND '{END_PERIOD}'
      AND work_schedule IN {WORK_SCHEDULES}
      AND home_yishuv_stat_2022 IS NOT NULL
    """)
    df_home["home_yishuv_stat_2022"] = (
        df_home["home_yishuv_stat_2022"].astype("string").str.replace(r"\.0$", "", regex=True)
    )
    # drop stale empty column if present
    out = out.drop(columns=["home_yishuv_stat_2022"], errors="ignore")
    out = out.merge(df_home, on=["identifier", "month"], how="left")
    print(
        f"Merged home SA for {out['home_yishuv_stat_2022'].notna().mean():.1%} of rows "
        f"({out['home_yishuv_stat_2022'].nunique():,} distinct SAs)."
    )
    return out


def _median_bin(series, labels=("Below median", "Above median")):
    """Split at the sample median; NaN stays missing. Ties at the median → Above."""
    out = pd.Series(pd.NA, index=series.index, dtype="string")
    valid = series.dropna()
    if len(valid) == 0:
        return out
    med = valid.median()
    out.loc[valid.index] = np.where(valid >= med, labels[1], labels[0])
    return out


def attach_census_occ(df):
    """Join home-SA IT / academic-occupation shares and median-split labels."""
    out = _ensure_home_yishuv(df)
    # avoid duplicate census cols on re-run
    out = out.drop(columns=["it_pct", "academic_occup_pct", "it_share_bin", "academic_occup_bin"],
                   errors="ignore")
    out = out.merge(df_census_occ, on="home_yishuv_stat_2022", how="left")
    out["it_share_bin"] = _median_bin(out["it_pct"])
    out["academic_occup_bin"] = _median_bin(out["academic_occup_pct"])
    return out


df_visits = attach_census_occ(df_visits)
df_weekday = attach_census_occ(df_weekday)
print(
    "Attached to weekday visits | "
    f"it_share_bin: {df_weekday['it_share_bin'].value_counts(dropna=False).to_dict()} | "
    f"academic_occup_bin: {df_weekday['academic_occup_bin'].value_counts(dropna=False).to_dict()}"
)


In [ ]:
# Feature sources shown on axis labels: Census 2022 (home SA %), Census tag
# (profile home-area classification), Mobile (GPS / dwell-derived).

FEATURE_SOURCE = {
    "Work location": "Mobile",
    "Employment zone": "Mobile",
    "Occupation proxy": "Census tag",
    "IT share": "Census 2022",
    "Academic occupation": "Census 2022",
    "Home area SES": "Census tag",
    "Religion (home area)": "Census tag",
    "Parent (School&Kindergarden visits)": "Mobile",
    "Student (University visits)": "Mobile",
    "Elderly (Medical visits)": "Mobile",
    "Remote worker": "Mobile",
    # "Likes cafes": "Mobile",
    # "Likes shopping": "Mobile",
    # "Likes outdoors": "Mobile",
}


In [ ]:



def dim_label(name):
    """Axis label with data source."""
    return f"{name} [{FEATURE_SOURCE[name]}]"


def build_dimensions(df):
    """Map each profile attribute onto readable group labels, dropping negligible levels.

    Education and car availability are omitted; IT share and academic occupation
    come from CBS Census 2022 home-SA percentages (same fields as notebook 3302),
    split at the median (below / above).
    """
    yes_no = {True: "Yes", False: "No"}
    return {
        dim_label("Work location"): df["work_context_type"].map(
            {"office": "Office", "residential": "Outside office zone"}),
        dim_label("Employment zone"): df["employment_zone"].map(yes_no),
        dim_label("Occupation proxy"): df["home_tag_occupation_proxy"].map(
            {"USR_OCCUPATION_KNOWLEDGE": "Knowledge", "USR_OCCUPATION_GENERAL": "General"}),
        dim_label("IT share"): df["it_share_bin"],
        dim_label("Academic occupation"): df["academic_occup_bin"],
        dim_label("Home area SES"): df["home_tag_ses"].map(
            {"USR_SES_LOW": "Low", "USR_SES_MED": "Medium", "USR_SES_HIGH": "High"}),
        dim_label("Religion (home area)"): df["home_tag_religion"].map(
            {"USR_RELIGION_SECULAR": "Secular", "USR_RELIGION_TRADITIONAL": "Traditional",
             "USR_RELIGION_RELIGIOUS": "Religious"}),
        dim_label("Parent (School&Kindergarden visits)"): df["FLAG_PARENT"].map(yes_no),
        dim_label("Student (University visits)"): df["FLAG_STUDENT"].map(yes_no),
        dim_label("Elderly (Medical visits)"): df["FLAG_MEDICAL"].map(yes_no),
        # dim_label("Religious visits"): df["FLAG_RELIGIOUS"].map(yes_no),
        # dim_label("Likes cafes"): df["like_cafes"].map(yes_no),
        # dim_label("Likes shopping"): df["like_shopping"].map(yes_no),
        # dim_label("Likes outdoors"): df["like_outdoor"].map(yes_no),
    }


dims_wd = build_dimensions(df_weekday)
DIM_ORDER = list(dims_wd.keys())

def composition_gap(df, dim_order=DIM_ORDER):
    """Remote minus on-site prevalence (pp) of each profile group among visits in `df`."""
    dims = build_dimensions(df)
    rows = []
    for dim, series in dims.items():
        sub = df.assign(_g=series).dropna(subset=["_g"])
        for sched in WORK_SCHEDULES:
            s = sub[sub["work_schedule"] == sched]
            if s.empty:
                continue
            share = s["_g"].value_counts(normalize=True)
            for group, val in share.items():
                rows.append({
                    "dimension": dim, "group": group,
                    "work_schedule": sched, "share": val, "n": len(s),
                })
    out = (pd.DataFrame(rows)
           .pivot_table(index=["dimension", "group"], columns="work_schedule", values="share")
           .reset_index())
    for sched in WORK_SCHEDULES:
        if sched not in out.columns:
            out[sched] = np.nan
    out["gap_pp"] = (out["remote_worker"] - out["onsite_worker"]) * 100
    out["dimension"] = pd.Categorical(out["dimension"], dim_order, ordered=True)
    binary_dims = {d for d, s in dims.items() if set(s.dropna().unique()) == {"Yes", "No"}}
    out = out[~(out["dimension"].isin(binary_dims) & (out["group"] == "No"))]
    out = out.sort_values(["dimension", "gap_pp"])
    out["label"] = np.where(
        out["dimension"].isin(binary_dims),
        out["dimension"].astype(str),
        out["dimension"].astype(str) + ": " + out["group"].astype(str),
    )
    return out, binary_dims


comp, BINARY_DIMS = composition_gap(df_weekday)

fig, ax = plt.subplots(figsize=(9, 7.5))
colors = ["#2b6cb0" if g > 0 else "#dd6b20" for g in comp["gap_pp"]]
ax.barh(comp["label"], comp["gap_pp"], color=colors)
ax.axvline(0, color="0.3", lw=1)
ax.invert_yaxis()
ax.set_xlabel("Difference in prevalence, remote minus on-site workers (percentage points)")
ax.set_ylabel("")
ax.set_title("Who visits detected third places: profile composition of remote versus on-site\n"
             "visitors, weekdays 2022-2023")
from matplotlib.patches import Patch
ax.legend(
    handles=[
        Patch(facecolor="#2b6cb0", label="More common among remote workers"),
        Patch(facecolor="#dd6b20", label="More common among on-site workers"),
    ],
    loc="lower right",
    frameon=True,
    fontsize=9,
)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.show()

caption("""**Fig. H.** Profile composition of weekday visitors to detected third places. Each bar is
the prevalence of a profile group among remote workers' visits minus its prevalence among on-site
workers' visits; blue means over-represented among remote workers. *Notes:* prevalence is visit-weighted;
axis labels show the data source — Census 2022 (home-SA IT share / academic occupation from CBS, as in
notebook 3302), Census tag (profile home-area classifications), Mobile (GPS / dwell-derived). Education
and car availability are omitted. For yes/no attributes only the "yes" side is shown; shares within a
multi-level dimension sum to zero across its levels.""")


In [ ]:
# Metric: profile-group prevalence gap (morning − early afternoon), percentage points
# Categories: Fig. H dimensions + Remote worker (Yes/No) as one more row
# X: gap (pp); stars = place-clustered (poi_id) test that gap ≠ 0

import statsmodels.api as sm
from matplotlib.patches import Patch

print(
    f"Metric: morning−early-afternoon prevalence gap (pp) | Categories: profile groups "
    f"+ Remote worker | X: gap | Significance: place-clustered SE (poi_id) | "
    f"Morning {MORNING_HOURS[0]:02d}:00–{MORNING_HOURS[1]:02d}:59 vs "
    f"early afternoon {EARLY_AFTERNOON_HOURS[0]:02d}:00–{EARLY_AFTERNOON_HOURS[1]:02d}:59"
)


def build_dimensions_h2(df):
    """Same as Fig. H, plus Remote worker as a Yes/No profile row."""
    dims = build_dimensions(df)
    dims[dim_label("Remote worker")] = df["work_schedule"].eq("remote_worker").map(
        {True: "Yes", False: "No"}
    )
    return dims


def place_clustered_gap_pvalue(df_window, series, group_level, place_col="poi_id"):
    """OLS of 1{group} on is_morning among morning∪early-afternoon visits; cluster by place.

    Coefficient on is_morning = P(group|morning) − P(group|early afternoon).
    Returns (p_value, se) for that gap (on the proportion scale, not pp).
    """
    sub = df_window.assign(_g=series).dropna(subset=["_g", place_col]).copy()
    if sub.empty or sub[place_col].nunique() < 2:
        return np.nan, np.nan
    y = (sub["_g"] == group_level).astype(float)
    # avoid perfect collinearity / empty cells
    if y.nunique() < 2 or sub["is_morning"].nunique() < 2:
        return np.nan, np.nan
    x = sm.add_constant(sub["is_morning"].astype(float))
    try:
        fit = sm.OLS(y, x).fit(
            cov_type="cluster",
            cov_kwds={"groups": sub[place_col].astype(str)},
        )
        return float(fit.pvalues["is_morning"]), float(fit.bse["is_morning"])
    except Exception:
        return np.nan, np.nan


def stars(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


DIM_ORDER_H2 = list(build_dimensions_h2(df_weekday).keys())
BINARY_DIMS_H2 = set(BINARY_DIMS) | {dim_label("Remote worker")}

# Pool morning + early afternoon once for clustered tests
df_h2 = df_weekday[df_weekday["is_morning"] | df_weekday["is_early_afternoon"]].copy()
dims_h2 = build_dimensions_h2(df_h2)

morning = df_h2[df_h2["is_morning"]]
early_aft = df_h2[df_h2["is_early_afternoon"]]
print(
    f"n_morning={len(morning):,} · n_early_afternoon={len(early_aft):,} · "
    f"n_places={df_h2['poi_id'].nunique():,}"
)

rows = []
for dim in DIM_ORDER_H2:
    series = dims_h2[dim]
    for window, mask in [
        ("morning", df_h2["is_morning"]),
        ("early_afternoon", df_h2["is_early_afternoon"]),
    ]:
        sub = df_h2.loc[mask].assign(_g=series[mask]).dropna(subset=["_g"])
        if sub.empty:
            continue
        share = sub["_g"].value_counts(normalize=True)
        for group, val in share.items():
            rows.append({
                "dimension": dim, "group": group, "window": window,
                "share": val, "n": len(sub),
            })

time_comp = (pd.DataFrame(rows)
             .pivot_table(index=["dimension", "group"], columns="window", values="share")
             .reset_index())
for col in ("morning", "early_afternoon"):
    if col not in time_comp.columns:
        time_comp[col] = np.nan
time_comp["gap_pp"] = (time_comp["morning"] - time_comp["early_afternoon"]) * 100
time_comp["dimension"] = pd.Categorical(time_comp["dimension"], DIM_ORDER_H2, ordered=True)
time_comp = time_comp[~(time_comp["dimension"].isin(BINARY_DIMS_H2) & (time_comp["group"] == "No"))]
time_comp = time_comp.sort_values(["dimension", "gap_pp"]).reset_index(drop=True)

# Place-clustered p-values for each plotted row
pvals, ses = [], []
for _, r in time_comp.iterrows():
    p, se = place_clustered_gap_pvalue(df_h2, dims_h2[r["dimension"]], r["group"])
    pvals.append(p)
    ses.append(se)
time_comp["p_place"] = pvals
time_comp["se_place"] = ses
time_comp["sig"] = time_comp["p_place"].lt(0.05).fillna(False)
time_comp["stars"] = time_comp["p_place"].map(stars)
time_comp["label"] = np.where(
    time_comp["dimension"].isin(BINARY_DIMS_H2),
    time_comp["dimension"].astype(str),
    time_comp["dimension"].astype(str) + ": " + time_comp["group"].astype(str),
)
time_comp["label_plot"] = time_comp["label"] + time_comp["stars"]

rw_lab = dim_label("Remote worker")
assert time_comp["label"].astype(str).str.startswith(rw_lab).any(), "Remote worker row missing"
print(
    f"Significant rows (place-clustered p<0.05): {int(time_comp['sig'].sum())} / {len(time_comp)}"
)
print(time_comp.loc[time_comp["label"].astype(str).str.startswith(rw_lab),
                    ["label", "gap_pp", "p_place", "stars"]].to_string(index=False))

y_pos = np.arange(len(time_comp))
fig, ax = plt.subplots(figsize=(9, 7.5))
colors = ["#2b6cb0" if g > 0 else "#dd6b20" for g in time_comp["gap_pp"]]
ax.barh(y_pos, time_comp["gap_pp"], color=colors)
ax.axvline(0, color="0.3", lw=1)
ax.set_yticks(y_pos)
ax.set_yticklabels(time_comp["label_plot"].tolist())
ax.invert_yaxis()
ax.set_xlabel(
    f"Difference in prevalence, morning ({MORNING_HOURS[0]:02d}:00–{MORNING_HOURS[1]:02d}:59) "
    f"minus early afternoon ({EARLY_AFTERNOON_HOURS[0]:02d}:00–{EARLY_AFTERNOON_HOURS[1]:02d}:59) (pp)"
)
ax.set_ylabel("")
ax.set_title(
    "Who visits detected third places: profile composition of morning versus\n"
    "early-afternoon visitors, weekdays 2022-2023"
)
ax.legend(
    handles=[
        Patch(facecolor="#2b6cb0", label="More common in the morning"),
        Patch(facecolor="#dd6b20", label="More common in early afternoon"),
    ],
    loc="lower right",
    frameon=True,
    fontsize=9,
    title="* p<0.05  ** p<0.01  *** p<0.001\n(place-clustered SE)",
    title_fontsize=8,
)
# Bold y labels when the place-clustered gap is significant
for tick, sig in zip(ax.get_yticklabels(), time_comp["sig"].tolist()):
    tick.set_fontweight("bold" if sig else "normal")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.show()

caption(f"""**Fig. H2.** Profile composition of weekday visitors to detected third places by arrival
window. Each bar is the prevalence of a profile group among morning visits
({MORNING_HOURS[0]:02d}:00–{MORNING_HOURS[1]:02d}:59) minus its prevalence among early-afternoon
visits ({EARLY_AFTERNOON_HOURS[0]:02d}:00–{EARLY_AFTERNOON_HOURS[1]:02d}:59); blue means
over-represented in the morning. Stars mark gaps that differ from zero at conventional levels
(* p<0.05, ** p<0.01, *** p<0.001) using OLS of the group indicator on a morning dummy with
**standard errors clustered by place (`poi_id`)**; significant rows are bold on the y-axis.
Remote worker is one Yes/No row. *Notes:* prevalence is visit-weighted; axis labels show the data
source; IT share and academic occupation are Census 2022 home-SA median splits; for yes/no
attributes only the "yes" side is shown.""")


In [ ]:
BEHAVIOUR_METRICS = {
    "is_early_afternoon": f"Arrives {EARLY_AFTERNOON_HOURS[0]}:00–{EARLY_AFTERNOON_HOURS[1]}:59 (%)",
    "is_long": "Stays longer than 1 hour (%)",
    "from_home": "Arrives directly from home (%)",
    "near_home": "Place is within 1 km of home (%)",
}

MIN_CELL = 200   # do not plot a group with fewer visits than this

behaviour_rows = []
for dim, series in dims_wd.items():
    sub = df_weekday.assign(_g=series).dropna(subset=["_g"])
    for metric in BEHAVIOUR_METRICS:
        r = share_by(sub, ["_g"], metric).rename(columns={"_g": "group"})
        r["dimension"], r["metric"] = dim, metric
        behaviour_rows.append(r)

behaviour = pd.concat(behaviour_rows, ignore_index=True)
behaviour = behaviour[behaviour["n"] >= MIN_CELL]
behaviour["dimension"] = pd.Categorical(behaviour["dimension"], DIM_ORDER, ordered=True)
behaviour["label"] = behaviour["dimension"].astype(str) + ": " + behaviour["group"].astype(str)

order = (behaviour[["dimension", "label"]].drop_duplicates()
         .sort_values(["dimension", "label"])["label"].tolist())

fig, axes = plt.subplots(1, 4, figsize=(18, 9), sharey=True)
dim_of_label = dict(zip(behaviour["label"], behaviour["dimension"].astype(str)))

for ax, (metric, title) in zip(axes, BEHAVIOUR_METRICS.items()):
    d = behaviour[behaviour["metric"] == metric].set_index("label").reindex(order)
    overall = df_weekday[metric].mean() * 100
    y = np.arange(len(d))
    # shade alternate dimensions so the reader can see which rows belong together
    for dim_i, dim in enumerate(DIM_ORDER):
        idx = [i for i, lab in enumerate(order) if dim_of_label.get(lab) == dim]
        if idx and dim_i % 2 == 0:
            ax.axhspan(min(idx) - 0.5, max(idx) + 0.5, color="0.85", alpha=0.45, zorder=0)
    ax.errorbar(d["share"] * 100, y,
                xerr=[(d["share"] - d["ci_low"]) * 100, (d["ci_high"] - d["share"]) * 100],
                fmt="o", ms=5, color="#2b6cb0", ecolor="#2b6cb0",
                elinewidth=1.4, capsize=2.5, zorder=3)
    ax.axvline(overall, ls="--", lw=1.2, color="#dd6b20", zorder=2,
               label=f"All visits ({overall:.1f}%)")
    ax.set_yticks(y)
    ax.set_yticklabels(order, fontsize=9)
    ax.set_ylim(len(d) - 0.5, -0.5)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Share of weekday visits (%)")
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(axis="y", visible=False)

axes[0].set_ylabel("")
fig.suptitle("Third-place behaviour by visitor profile, weekdays 2022-2023", y=0.997)
plt.tight_layout()
plt.show()

caption("""**Fig. I.** Four behavioural measures of third-place use, broken down by every usable
attribute in the monthly user profile, weekdays 2022-2023. Points are shares of visits with 95%
Wilson binomial confidence intervals; the dashed line marks the overall average for each measure
and grey banding groups the levels of one profile dimension. *Notes:* the horizontal axis is not
anchored at zero, so differences between groups are magnified; profile groups with fewer than 200
visits are omitted; amenity affinity uses like_cafes / like_shopping rebuilt from non-third-place dwells.""")


In [ ]:
HOURLY_SPLITS = [
    ("work_schedule",
     {"remote_worker": "Remote worker", "onsite_worker": "On-site worker"},
     "Work schedule"),
    # ("work_context_type",
    #  {"office": "Works at an office",
    #   "residential": "Not office",
    #   "other": "Not office",
    #   "public_space": "Not office"},
    #  "Work location type"),
    ("home_tag_occupation_proxy",
     {"USR_OCCUPATION_KNOWLEDGE": "Knowledge occupation",
      "USR_OCCUPATION_GENERAL": "General occupation"},
     "Occupation proxy"),
    ("FLAG_PARENT", {True: "Parent (school runs)", False: "No school runs"}, "Parenting"),
   
]

fig, axes = plt.subplots(1, len(HOURLY_SPLITS), figsize=(4.0 * len(HOURLY_SPLITS), 3.6), sharey=True)

for ax, (col, mapping, title) in zip(axes, HOURLY_SPLITS):
    sub = df_weekday.assign(_g=df_weekday[col].map(mapping)).dropna(subset=["_g"])
    totals = sub.groupby("_g").size()
    d = share_by(sub.assign(one=True), ["_g", "hour"], "one")
    d["n"] = d["_g"].map(totals)
    d = add_wilson_ci(d)
    for (group, gd), color in zip(d.groupby("_g"), ["#2b6cb0", "#dd6b20"]):
        gd = gd.sort_values("hour")
        ax.plot(gd["hour"], gd["share"] * 100, marker="o", ms=3.5, color=color,
                label=f"{group} (n={totals[group]:,})")
        ax.fill_between(gd["hour"], gd["ci_low"] * 100, gd["ci_high"] * 100,
                        color=color, alpha=0.20)
    ax.axvspan(EARLY_AFTERNOON_HOURS[0] - 0.5, EARLY_AFTERNOON_HOURS[1] + 0.5, color="grey", alpha=0.10, zorder=0)
    ax.set_title(title)
    ax.set_xlabel("Hour of arrival (local time)")
    ax.set_xticks(range(0, 24, 2))
    ax.legend(fontsize=8, loc="upper left")

axes[0].set_ylabel("Share of the group's visits (%)")
fig.suptitle("Weekday hourly rhythm of third-place visits by visitor profile, 2022-2023", y=1.02)
plt.show()

caption("""**Fig. J.** Weekday hourly distribution of third-place visits for three profile
splits: work schedule (remote vs on-site), office vs not-office work location, knowledge vs
general occupation, parenting (school runs), and student flag. Shaded band marks 10:00-15:00; bands around the lines are 95%
Wilson binomial confidence intervals. *Notes:* FLAG_STUDENT is based on repeated visits to
university/college POIs in the same month; the student cell is small relative to other splits.""")

In [ ]:
# HOURLY_SPLITS = [
#     ("work_schedule",
#      {"remote_worker": "Remote worker", "onsite_worker": "On-site worker"},
#      "Work schedule"),
#     ("work_context_type",
#      {"office": "Works at an office",
#       "residential": "Not office",
#       "other": "Not office",
#       "public_space": "Not office"},
#      "Work location type"),
#     ("home_tag_occupation_proxy",
#      {"USR_OCCUPATION_KNOWLEDGE": "Knowledge occupation",
#       "USR_OCCUPATION_GENERAL": "General occupation"},
#      "Occupation proxy"),
#     ("FLAG_PARENT", {True: "Parent (school runs)", False: "No school runs"}, "Parenting"),
#     ("FLAG_STUDENT", {True: "Student", False: "Not a student"}, "Student"),
# ]

fig, axes = plt.subplots(1, len(HOURLY_SPLITS), figsize=(4.0 * len(HOURLY_SPLITS), 4.6), sharey=True)

for ax, (col, mapping, title) in zip(axes, HOURLY_SPLITS):
    sub = df_weekday.assign(_g=df_weekday[col].map(mapping)).dropna(subset=["_g"])
    totals = sub.groupby("_g").size()
    d = share_by(sub.assign(one=True), ["_g", "hour"], "one")
    d["n"] = d["_g"].map(totals)
    d = add_wilson_ci(d)
    for (group, gd), color in zip(d.groupby("_g"), ["#2b6cb0", "#dd6b20"]):
        gd = gd.sort_values("hour")
        ax.plot(gd["hour"], gd["share"] * 100, marker="o", ms=3.5, color=color,
                label=f"{group} (n={totals[group]:,})")
        ax.fill_between(gd["hour"], gd["ci_low"] * 100, gd["ci_high"] * 100,
                        color=color, alpha=0.20)
    ax.axvspan(EARLY_AFTERNOON_HOURS[0] - 0.5, EARLY_AFTERNOON_HOURS[1] + 0.5, color="grey", alpha=0.10, zorder=0)
    ax.set_title(title)
    ax.set_xlabel("Hour of arrival (local time)")
    ax.set_xticks(range(0, 24, 2))
    ax.legend(fontsize=8, loc="upper left")

axes[0].set_ylabel("Share of the group's visits (%)")
fig.suptitle("Weekday hourly rhythm of third-place visits by visitor profile, 2022-2023", y=1.02)
plt.show()

caption("""**Fig. J.** Weekday hourly distribution of third-place visits for three profile
splits: work schedule (remote vs on-site), office vs not-office work location, knowledge vs
general occupation, parenting (school runs), and student flag. Shaded band marks 10:00-15:00; bands around the lines are 95%
Wilson binomial confidence intervals. *Notes:* FLAG_STUDENT is based on repeated visits to
university/college POIs in the same month; the student cell is small relative to other splits.""")

In [ ]:
# Metric: share of the group's visits by hour of arrival
# Categories: same HOURLY_SPLITS as Fig. J (schedule, office, occupation, parenting, student)
# X: hour of arrival; one figure per POI primarytype with enough weekday visits
MIN_VISITS_PER_TYPE = 800
MIN_GROUP_VISITS = 80
# HOURLY_SPLITS = [
#     ("work_schedule",
#      {"remote_worker": "Remote worker", "onsite_worker": "On-site worker"},
#      "Work schedule"),
#     ("work_context_type",
#      {"office": "Works at an office",
#       "residential": "Not office",
#       "other": "Not office",
#       "public_space": "Not office"},
#      "Work location type"),
#     ("home_tag_occupation_proxy",
#      {"USR_OCCUPATION_KNOWLEDGE": "Knowledge occupation",
#       "USR_OCCUPATION_GENERAL": "General occupation"},
#      "Occupation proxy"),
#     ("FLAG_PARENT", {True: "Parent (school runs)", False: "No school runs"}, "Parenting"),
#     ("FLAG_STUDENT", {True: "Student", False: "Not a student"}, "Student"),
# ]

type_counts = df_weekday["primarytype_agg"].value_counts()
poi_types = type_counts[type_counts >= MIN_VISITS_PER_TYPE].index.tolist()
print("POI types plotted:", {t: int(type_counts[t]) for t in poi_types})

for ptype in poi_types:
    df_type = df_weekday[df_weekday["primarytype_agg"] == ptype].copy()
    fig, axes = plt.subplots(1, len(HOURLY_SPLITS), figsize=(4.0 * len(HOURLY_SPLITS), 4.6), sharey=True)

    for ax, (col, mapping, title) in zip(axes, HOURLY_SPLITS):
        sub = df_type.assign(_g=df_type[col].map(mapping)).dropna(subset=["_g"])
        totals = sub.groupby("_g").size()
        # skip a panel series if the cell is tiny, but still draw what we can
        keep = [g for g, n in totals.items() if n >= MIN_GROUP_VISITS]
        sub = sub[sub["_g"].isin(keep)]
        if sub.empty or len(keep) < 2:
            ax.set_title(f"{title}\n(insufficient sample)")
            ax.set_xlabel("Hour of arrival (local time)")
            continue

        totals = sub.groupby("_g").size()
        d = share_by(sub.assign(one=True), ["_g", "hour"], "one")
        d["n"] = d["_g"].map(totals)
        d = add_wilson_ci(d)
        for (group, gd), color in zip(d.groupby("_g"), ["#2b6cb0", "#dd6b20"]):
            gd = gd.sort_values("hour")
            ax.plot(gd["hour"], gd["share"] * 100, marker="o", ms=3.5, color=color,
                    label=f"{group} (n={totals[group]:,})")
            ax.fill_between(gd["hour"], gd["ci_low"] * 100, gd["ci_high"] * 100,
                            color=color, alpha=0.20)
        ax.axvspan(EARLY_AFTERNOON_HOURS[0] - 0.5, EARLY_AFTERNOON_HOURS[1] + 0.5, color="grey", alpha=0.10, zorder=0)
        ax.set_title(title)
        ax.set_xlabel("Hour of arrival (local time)")
        ax.set_xticks(range(0, 24, 2))
        ax.legend(fontsize=8, loc="upper left")

    axes[0].set_ylabel("Share of the group's visits (%)")
    fig.suptitle(
        f"Weekday hourly rhythm by visitor profile — {ptype.replace('_', ' ')} "
        f"(n={len(df_type):,} visits), 2022-2023",
        y=1.02,
    )
    plt.show()

caption(f"""**Fig. L.** Same three profile splits as Fig. J, repeated for each POI primary type
with at least {MIN_VISITS_PER_TYPE} weekday visits ({', '.join(poi_types)}). Within each type,
profile groups with fewer than {MIN_GROUP_VISITS} visits are omitted. Shaded band marks
10:00-15:00; bands around the lines are 95% Wilson binomial confidence intervals.""")


In [ ]:
import statsmodels.formula.api as smf

model_cols = ["is_early_afternoon", "is_long", "from_home", "near_home", "identifier", "work_schedule",
              "work_context_type", "employment_zone", "home_tag_occupation_proxy",
              "home_tag_education", "home_tag_mobility", "home_tag_ses",
              "FLAG_PARENT", "FLAG_STUDENT", "FLAG_MEDICAL",
              "like_cafes", "like_shopping", "like_outdoor", "primarytype"]

df_model = df_weekday[model_cols].copy()
df_model = df_model[df_model["work_context_type"].isin(["office", "residential"])].dropna()
for col in ["is_early_afternoon", "is_long", "from_home", "near_home", "employment_zone", "FLAG_PARENT",
            "FLAG_STUDENT", "FLAG_MEDICAL", "like_cafes", "like_shopping", "like_outdoor"]:
    df_model[col] = df_model[col].astype(int)

RHS = ("C(work_schedule, Treatment('onsite_worker'))"
       " + C(work_context_type, Treatment('office'))"
       " + employment_zone"
       " + C(home_tag_occupation_proxy, Treatment('USR_OCCUPATION_GENERAL'))"
       " + C(home_tag_education, Treatment('USR_EDU_GENERAL'))"
       " + C(home_tag_mobility, Treatment('USR_CAR_AVG'))"
       " + C(home_tag_ses, Treatment('USR_SES_MED'))"
       " + FLAG_PARENT + FLAG_STUDENT + FLAG_MEDICAL"
       " + like_cafes + like_shopping + like_outdoor"
       " + C(primarytype)")

PRETTY = {
    "C(work_schedule, Treatment('onsite_worker'))[T.remote_worker]": "Remote worker",
    "C(work_context_type, Treatment('office'))[T.residential]": "Works from home",
    "employment_zone": "Works in an employment zone",
    "C(home_tag_occupation_proxy, Treatment('USR_OCCUPATION_GENERAL'))[T.USR_OCCUPATION_KNOWLEDGE]":
        "Knowledge occupation",
    "C(home_tag_education, Treatment('USR_EDU_GENERAL'))[T.USR_EDU_ACADEMIC]": "Academic education",
    "C(home_tag_education, Treatment('USR_EDU_GENERAL'))[T.USR_EDU_BACHELOR]": "Bachelor education",
    "C(home_tag_mobility, Treatment('USR_CAR_AVG'))[T.USR_CAR_HIGH]": "High car availability",
    "C(home_tag_mobility, Treatment('USR_CAR_AVG'))[T.USR_CAR_LOW]": "Low car availability",
    "C(home_tag_ses, Treatment('USR_SES_MED'))[T.USR_SES_HIGH]": "High SES home area",
    "C(home_tag_ses, Treatment('USR_SES_MED'))[T.USR_SES_LOW]": "Low SES home area",
    "FLAG_PARENT": "Parent (school runs)",
    "FLAG_STUDENT": "Student",
    "FLAG_MEDICAL": "Frequent medical visits",
    "like_cafes": "Frequents cafes elsewhere",
    "like_shopping": "Frequents retail elsewhere",
    "like_outdoor": "Frequents parks elsewhere",
}

results = {}
for outcome in ["is_early_afternoon", "is_long", "from_home", "near_home"]:
    fit = smf.logit(f"{outcome} ~ {RHS}", data=df_model).fit(
        disp=False, cov_type="cluster", cov_kwds={"groups": df_model["identifier"]})
    odds = np.exp(fit.params)
    results[BEHAVIOUR_METRICS[outcome].replace(" (%)", "")] = pd.Series(
        [f"{odds[k]:.2f}{'*' if fit.pvalues[k] < 0.05 else ''}" if k in odds else ""
         for k in PRETTY], index=PRETTY.values())

odds_table = pd.DataFrame(results)

### What the profiles add

[`002-users-profiles-dynamic.ipynb`](002-users-profiles-dynamic.ipynb) from
`'food'` and `'retail'` evidently did not match there — in `dwells_pois_home_work_v2` they match
578k and 763k dwells respectively, so the definitions would work if re-run against v2. The notebook
rebuilds both flags rather than using the stored columns. Separately, `employment_zone` is a 0/1
indicator, not a zone identifier, so it cannot be used to compare individual employment zones.
`FLAG_RELIGIOUS` (0.17% of visits) and `home_tag_lifecycle` (94% one level) are too concentrated to
support a split.

**Third-place users are a selected group, and remote workers doubly so.** Remote visitors are drawn
disproportionately from knowledge occupations (+3.8 pp), degree-educated home areas (+6.5 pp
combined) and high-SES home areas (+5.4 pp), and they are markedly *less* likely to be parents
(-4.7 pp). This is the classic critique of Oldenburg's account: the "great good place" is not
equally available to everyone, and remote work appears to widen rather than narrow that gradient.
The largest single compositional gap is amenity affinity — remote visitors are 6.2 pp more likely
to be frequent retail users and 3.3 pp more likely to be frequent cafe users elsewhere in the same
month, so third-place use is part of a broader pattern of out-of-home activity rather than a
substitute for it.

**Constraints shape timing far more than tastes do.** In the pooled model, being a parent who does
school runs cuts the odds of an early-afternoon arrival by 23% and being a student by 25% — larger effects
than remote work itself (+16%). Figure J makes the mechanism visible: parents' visits collapse in
the 10:00–13:59 window and rebound into a pronounced 17:00-20:00 peak, the shape of a day organised
around school hours. Oldenburg's "third place" presumes discretionary time; these profiles show who
actually has it.

**Car availability, not income, governs locality.** High car availability cuts the odds that the
third place is within 1 km of home by a third (OR 0.67), the strongest locality effect in the
model, while high-SES home areas show only a weak negative effect (0.87) and low car availability
raises the odds of a long stay by 28%. Walkable third-place use is a function of mobility
constraint, and people who cannot drive both stay closer and stay longer once they arrive.

**The `work_context_type` label adds nothing once `work_schedule` is controlled.** Users whose
detected work location is residential rather than an office show no significant difference on any
of the four outcomes (odds ratios 0.84-1.15, all non-significant), while the `work_schedule` label
remains strongly predictive of arriving from home (OR 1.71) and of proximity to home (OR 1.69). The
two variables are measuring the same underlying phenomenon and `work_schedule` measures it better;
there is no reason to carry both into later models. Working in an employment zone does have an
independent effect on timing (OR 0.79 for early-afternoon arrivals), which is what one would expect of
people tied to a fixed workplace.

**Caveat.** All models are estimated over visits, not users, so people who visit often carry more
weight; standard errors are clustered by user to account for this, but the point estimates still
describe the average *visit* rather than the average *person*. The home-area tags are
characteristics of the statistical area, not verified attributes of the individual, so they are
subject to ecological inference and should be read as neighbourhood context.


## 7. Summary of findings

Formal comparison of remote and on-site workers on the behavioural dimensions the third-place
literature treats as diagnostic.

In [ ]:
a = df_weekday[df_weekday["work_schedule"] == "remote_worker"]
b = df_weekday[df_weekday["work_schedule"] == "onsite_worker"]

# early afternoon 10–13 + late afternoon 14–18 (same buckets as 005 / pois_visits_metrics)
AFTERNOON_HOURS = range(EARLY_AFTERNOON_HOURS[0], LATE_AFTERNOON_HOURS[1] + 1)  # 10–18
a_aft = a[a["hour"].isin(AFTERNOON_HOURS)]
b_aft = b[b["hour"].isin(AFTERNOON_HOURS)]

rows = []
x, y = a_aft["duration_hours"].dropna(), b_aft["duration_hours"].dropna()
_, p = stats.mannwhitneyu(x, y, alternative="two-sided")
rows.append({"metric": "Mean visit duration (weekday 10:00–18:59)", "unit": "hours",
             "remote_workers": x.median(), "onsite_workers": y.median(),
             "difference": x.median() - y.median(), "p_value": p, "test": "Mann-Whitney U"})

x, y = a["dist_home_km"].dropna(), b["dist_home_km"].dropna()
_, p = stats.mannwhitneyu(x, y, alternative="two-sided")
rows.append({"metric": "Mean distance from home", "unit": "km",
             "remote_workers": x.mean(), "onsite_workers": y.mean(),
             "difference": x.mean() - y.mean(), "p_value": p, "test": "Mann-Whitney U"})

for label, col, sample_a, sample_b in [
    ("Visits longer than 1 hour (weekday 10:00–18:59)", "is_long", a_aft, b_aft),
    ("Visits longer than 1 hour (all weekday)", "is_long", a, b),
    ("Visits arriving from home", "from_home", a, b),
    ("Visits arriving from work", "from_work", a, b),
    (f"Visits arriving {EARLY_AFTERNOON_HOURS[0]}:00–{EARLY_AFTERNOON_HOURS[1]}:59", "is_early_afternoon", a, b),
    ("Visits within 1 km of home", None, a, b),
]:
    if col is None:
        x = (sample_a["dist_home_km"] < 1).dropna()
        y = (sample_b["dist_home_km"] < 1).dropna()
    else:
        x, y = sample_a[col], sample_b[col]
    table = np.array([[x.sum(), len(x) - x.sum()], [y.sum(), len(y) - y.sum()]])
    _, p, _, _ = stats.chi2_contingency(table)
    rows.append({"metric": label, "unit": "% of visits", "remote_workers": x.mean() * 100,
                 "onsite_workers": y.mean() * 100,
                 "difference": (x.mean() - y.mean()) * 100,
                 "p_value": p, "test": "Chi-square"})

summary = pd.DataFrame(rows).round({"remote_workers": 2, "onsite_workers": 2, "difference": 2})
summary["p_value"] = summary["p_value"].map(lambda p: f"{p:.1e}" if p < 0.001 else f"{p:.3f}")

### Reading of the results

The model detects **550 third places** out of 3,524 scored POIs. They are overwhelmingly food and
drink venues (155 coffee shops, 110 cafes, 52 bakeries) alongside a civil tier of cultural centres,
libraries and community centres (125 together). Location is now split by Tel Aviv-Yafo (`semel_yishuv` = 5000) versus the rest of Gush Dan;
re-run the crosstab and Fig. F for the updated counts.

The visit panel covers **60,886 visits by 31,240 users at 496 places**. Four results speak to the
claims the third-place literature treats as diagnostic.

**1. Locality — the strongest result.** Remote workers' third places are markedly closer to home:
4.13 km versus 5.34 km on average, and **36.6% of their visits are within 1 km of home against
25.1% for on-site workers** (both p < 0.001). Figure C shows the gap holds at every hour of the
working day and closes only late in the evening, when both groups travel further. The same pattern
appears in the paired place-level test, where the same POI draws remote workers from 0.82 km closer
than its average visitor. This is the clearest evidence that remote work pulls third-place use down
to the neighbourhood scale.

**2. Embedding in the domestic routine.** **31.7% of remote workers' weekday visits begin directly
at home, against 21.4% for on-site workers**, while arrivals from work run the other way (8.6%
versus 16.5%). For commuters the third place is attached to the journey to or from work; for remote
workers it is an excursion from the home. Figure E shows the gap opening from mid-morning onwards
and widening through the evening.

**3. Timing — a modest shift, not a wholesale one.** Remote workers do tilt into the working day
(47.2% of their visits arrive between 10:00 and 15:00 versus 43.1%, p < 0.001), and the place-level
paired test confirms a higher early-afternoon share. But Figure A shows both groups share the same
double-peaked rhythm, and the difference is a few percentage points rather than a distinct regime.
These places are not being converted into offices; they are absorbing some additional early-afternoon use.

**4. Lingering — the expected result does not hold.** Even restricting to weekday early and late
afternoon (10:00-18:00), remote workers' visits are slightly *shorter* (0.68 h versus 0.74 h;
21.6% versus 24.1% exceed one hour, p < 0.001). The same gap appears across all weekday hours
(0.72 h versus 0.79 h) and in the paired place-level test. Figure D shows on-site workers ahead at
nearly every hour. Two readings are available: either remote workers use these places for short
breaks from home rather than as work locations, or the dwell-detection pipeline truncates the long
sessions that would identify genuine remote-work visits. Separating these requires the hourly
signal data rather than dwell aggregates, and it matters because the PU model treats long weekday
visits as a positive signal.

**5. Context.** Same-category supply is highly skewed (median 7 POIs within 1 km, mean 21, max 113)
and tracks the urban structure (see Fig. F for Tel Aviv versus the rest of Gush Dan). Location-group differences in visitor mix should be read from the updated Fig. F. Lingering rises modestly
with density (22.6% to 27.1% of visits over an hour from the low to high tertile), suggesting that
dense amenity clusters support longer stays even though they draw visitors from further away
(6.6 km versus 5.0 km).

**Caveat on the visitor-mix figures.** The visit panel keeps only users whose schedule could be
determined, so remote workers are about half of it by construction. Remote workers are roughly 19%
of all weekday visits at the average detected place. Levels in Figures F and G are therefore not
population shares; only differences between groups should be read.
